# The Arcade Playbook | Coach on Stall

Based on [GGA LB 0.360 by Kozykappa](https://www.kaggle.com/code/kozykappa/gga-lb-0-360?scriptVersionId=375211512). The 0.360 score belongs to the upstream author. This snapshot includes the v4 reasoning coach.

The agent crops the game area, lets timed programs execute for up to 1.5 seconds, and credits screen changes to the program that was running.

Suggested version name `--try-coach-on-stall`.

## Coach action after a repeated input

A recent coach action can interrupt a low-impact loop when it differs from the repeated command. A rejected input, a scene change, or a changed coordinate crop prevents use of stale advice. A timed program retains its existing priority. Logs label the extra intervention `coach-loop`.

The intervention is inspired by [GGA Qwen Coach CA2](https://www.kaggle.com/code/alexchilton/gga-qwen-coach-ca2).

## Upstream credits

- the four sample games: **Tong Hui Kang's scripted pixel solvers** ([GGA Public Solver](https://www.kaggle.com/code/huikang/gga-public-solver), copied unchanged), picked by `obs.info["game_id"]` or the brief;
- every other game: **our start step** (pixel perception + the inputs the brief names, until the game leaves its opening screen) in front of the **QwenFN agent** by parthenos ([GGA: QwenFN agent](https://www.kaggle.com/code/nihilisticneuralnet/gga-qwenfn-agent)), with **lightningv08's background coach** ([0.294 | Qwen3.8-Flash-Next 125B + Background Coach](https://www.kaggle.com/code/lightningv08/0-294-qwen3-8-flash-next-125b-background-coach)): the planner runs on a background thread, one request at a time, and replies older than 12 s of game time are dropped.


## GGA: start step + any model agent + scripted sample-game solvers

Each game goes to exactly one agent.
The four sample games go to Tong Hui Kang's scripted pixel solvers ([GGA Public Solver](https://www.kaggle.com/code/huikang/gga-public-solver)).
Every other game goes to our start step (it clicks through title screens in code), then to the model agent in `MODEL_AGENT`.
The model agent here is nihilisticneuralnet's QwenFN planner/actor ([GGA: QwenFN agent](https://www.kaggle.com/code/nihilisticneuralnet/gga-qwenfn-agent)) on dfranzen's Qwen3.8-Flash-Next serving stack, unchanged.


## Credit: the general agent

The general agent for the hidden games, and the cells that set it up, serve it, play a sample game and submit, are
from **[GGA: QwenFN agent](https://www.kaggle.com/code/nihilisticneuralnet/gga-qwenfn-agent?scriptVersionId=373376666)** by **nihilisticneuralnet** (Version 4), used unchanged apart from a source
comment at the top of each cell and the play cell driving the base agent directly. That notebook in turn reuses
dfranzen's Qwen3.8-Flash-Next serving stack from the ARC-AGI-3 milestone-2 solution (built on Tufa Labs' Duck
harness). The scripted solvers for the four sample games and the routing agent are this notebook's own.

## GGA: Qwen3.8-Flash-Next planner/actor agent

Planner/actor agent served by **Qwen3.8-Flash-Next** (125B MoE, 6B active) through the Pennyroyal SGLang fork
(v2.5.3) with the albucino MTP drafter. This is the stack from dfranzen's ARC-AGI-3 milestone-2 solution (built
on Tufa Labs' Duck harness); its install and launch code is reused unchanged except for sizing and the interpreter.

**Startup:** the server installs and loads in the background while Chromium installs. The serve cell waits for it
until 12.5 min after notebook start, then serves regardless (limit: 15 min); until the model answers, the agent
plays cheap scripted probes. If the server dies mid-run it is relaunched (up to twice). Log: `/kaggle/working/serve.log`.


## Config and common setup

In [ ]:


import os, sys, time
NOTEBOOK_START_TIME = time.time()
os.environ.setdefault('PYTHONHASHSEED', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import contextlib, io, json, random, subprocess, threading, warnings
from pathlib import Path

RERUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in {'1', 'true'}
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)


SERVE_AT_S = 12.5 * 60


root = Path('/kaggle/input')


def _find_dir(preferred, *patterns):
    if preferred and Path(preferred).is_dir():
        return Path(preferred)
    for pat in patterns:
        hits = sorted({p.parent for p in root.rglob(pat)})
        if hits:
            return hits[0]
    return None


WHEELHOUSE_DIR = _find_dir('/kaggle/input/datasets/dfranzen/pennyroyal-v253')
if WHEELHOUSE_DIR is None:
    WHEELHOUSE_DIR = next((p for p in sorted(root.glob('**/*pennyroyal*'))
                           if p.is_dir() and any(p.rglob('requirements.lock'))), None)
FLASH_MODEL_DIR = _find_dir('/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1',
                            '*flash-next-w4a16*/**/model.safetensors.index.json')
FLASH_DRAFT_DIR = _find_dir('/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1',
                            '*flash-next-drafter*/**/model.safetensors.index.json')
SERVED_MODEL_NAME = 'flashnext'
SERVED_MODEL_PORT = 8001
FLASH_URL = f'http://127.0.0.1:{SERVED_MODEL_PORT}'

harness = next(root.rglob('gga/protocol.py')).parents[1]
kit = next(root.rglob('kit/wheels')).parent
print(f'rerun={RERUN}\nwheelhouse={WHEELHOUSE_DIR}\nflash={FLASH_MODEL_DIR}\ndraft={FLASH_DRAFT_DIR}')
if not (WHEELHOUSE_DIR and FLASH_MODEL_DIR):
    raise RuntimeError('Attach dfranzen/pennyroyal-v253 and dfranzen/intel-qwen3.8-flash-next-w4a16-autoround')
if FLASH_DRAFT_DIR is None:
    print('warning: MTP drafter not attached -> no speculative decoding (slower replies)')


chromium = subprocess.Popen(f'cp -r {kit}/browsers /tmp && chmod -R +x /tmp/browsers'
                            f' && dpkg -i {kit}/debs/*.deb > /dev/null 2>&1', shell=True)
subprocess.run(f'pip install -q --no-index --find-links {kit}/wheels playwright',
               shell=True, check=True)
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1920x1080x24'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(harness))
os.environ.update(GGA_ROOT=str(harness), GGA_SEED='0', DISPLAY=':99', PLAYWRIGHT_BROWSERS_PATH='/tmp/browsers')

import numpy as np
random.seed(0)
np.random.seed(0)
warnings.filterwarnings('ignore')
print(f'common setup done at {time.time() - NOTEBOOK_START_TIME:.0f}s')


## Launch Qwen3.8-Flash-Next (SGLang, background)

In [ ]:







import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error, threading, signal
import contextlib, copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

PREFIX = "/tmp/sgl-intel"
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=32 * 1024,
    MEMFRAC=0.94,
    MAXREQ=2,
    CUDAGRAPH_MAXBS=2,
    MAMBA_CACHE=16,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=FLASH_DRAFT_DIR is not None,
    FRSPEC=True,
    AUTOTUNE=True,
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]


    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view




def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024

    chunks = []
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')



def _py_version(exe):
    try:
        out = subprocess.run([exe, "-c", "import sys, sysconfig, os; print(sys.version_info[0], sys.version_info[1], "
                              "int(os.path.isfile(os.path.join(sysconfig.get_paths()['include'], 'Python.h'))))"],
                             text=True, capture_output=True, timeout=30).stdout.split()
        return int(out[0]), int(out[1]), bool(int(out[2]))
    except Exception:
        return None


def _usable312(exe):
    v = _py_version(exe) if exe and os.path.isfile(exe) else None
    return v is not None and v[:2] == (3, 12) and v[2]


def find_python312():
    """A CPython 3.12 with headers: on the image, or a python-build-standalone build attached as an input."""
    cands = [shutil.which("python3.12"), "/usr/bin/python3.12", "/usr/local/bin/python3.12", "/opt/conda/bin/python3.12",
             *sorted(glob.glob("/opt/conda/envs/*/bin/python3.12")),
             *sorted(glob.glob("/root/.local/share/uv/python/cpython-3.12*/bin/python3.12")),
             "/tmp/py312/python/bin/python3.12"]
    for exe in cands:
        if _usable312(exe):
            return exe


    dest = Path("/tmp/py312")
    tars = sorted(p for p in Path("/kaggle/input").rglob("cpython-3.12*x86_64*linux-gnu*.tar.gz"))
    dirs = sorted(p.parents[1] for p in Path("/kaggle/input").rglob("python3.12")
                  if p.parent.name == "bin" and (p.parents[1] / "include").is_dir())
    if tars:
        import tarfile
        shutil.rmtree(dest, ignore_errors=True)
        dest.mkdir(parents=True)
        with tarfile.open(tars[0]) as tf:
            tf.extractall(dest, filter="tar") if sys.version_info >= (3, 12) else tf.extractall(dest)
    elif dirs:
        shutil.rmtree(dest, ignore_errors=True)
        shutil.copytree(dirs[0], dest / "python", symlinks=True)
    else:
        raise RuntimeError(
            "No Python 3.12 found. The Pennyroyal wheelhouse only has cp312 wheels. Attach a dataset with "
            "python-build-standalone 'cpython-3.12.<x>+<date>-x86_64-unknown-linux-gnu-install_only.tar.gz' "
            "(github.com/astral-sh/python-build-standalone/releases), or pin the notebook to a Python 3.12 Kaggle image.")
    exe = dest / "python/bin/python3.12"
    for p in (dest / "python/bin").iterdir():
        with contextlib.suppress(Exception):
            p.chmod(0o755)
    if not _usable312(str(exe)):
        raise RuntimeError(f"Attached Python 3.12 at {exe} does not run or lacks headers")
    return str(exe)


PY312 = find_python312()
print("server interpreter:", PY312, _py_version(PY312))


def _prepare_flashnext(MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR):

    MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
    assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
    assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
    assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0

    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
            raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
    Path(PREFIX).mkdir(parents=True, exist_ok=True)
    Path(LOG).parent.mkdir(parents=True, exist_ok=True)
    WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
    LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
    sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
    assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
    wheel = sglang_wheels[0]
    install_marker = Path(PREFIX) / "installed-bundle.txt"


    install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
    install_env.pop("PYTHONPATH", None)
    if not Path(SGLANG).exists() or not install_marker.is_file() or (_py_version(PYTHON) or (0, 0))[:2] != (3, 12):
        print("=== installing Pennyroyal (offline) ===")

        uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
        assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
        uv_binary = Path(PREFIX) / "bin/uv"
        uv_binary.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(uv_wheels[0]) as archive:
            members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
            assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
            binary = archive.read(members[0])
            assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
            uv_binary.write_bytes(binary)
        uv_binary.chmod(0o755)
        uv = [str(uv_binary)]
        if Path(PYTHON).exists() and (_py_version(PYTHON) or (0, 0))[:2] != (3, 12):
            shutil.rmtree(VENV, ignore_errors=True)
        if not Path(PYTHON).exists():
            run(uv + ["venv", "--python", PY312, VENV], env=install_env)
        install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
        run(install + ["-r", str(LOCK)], env=install_env)
        run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
        install_marker.write_text('install')
    else:
        print(f"Matching offline bundle already installed at {VENV}")


    cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
    if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
        raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
    CUDA_HOME = str(cu13[0])
    libdir = Path(CUDA_HOME) / "lib"
    lib64 = Path(CUDA_HOME) / "lib64"
    if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
    for so in sorted(libdir.glob("*.so.*")):
        link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
        if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
    assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
    if not (libdir / "libcuda.so").exists():
        candidates = []
        for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
            candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
        if not candidates:
            code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
            maps = run([PYTHON, "-c", code], env=install_env).stdout
            candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
        if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
        link = libdir / "libcuda.so"
        if link.is_symlink(): link.unlink()
        link.symlink_to(candidates[0].resolve())
    CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
    assert CXX, "Host C++ compiler not found"
    CC = CXX.replace("g++", "gcc")
    assert shutil.which(CC), f"Matching C compiler missing: {CC}"


    env = dict(os.environ)
    env.pop("PYTHONPATH", None)
    cache = f"{PREFIX}/cache"
    for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
        Path(cache, name).mkdir(parents=True, exist_ok=True)
    env.update({
        "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
        "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
        "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
        "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
        "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
        "TORCH_CUDA_ARCH_LIST": "12.0",
        "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
        "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
        "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
        "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
        "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
        "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
        "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
        "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
        "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
        "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
        "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
        "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
        "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
    })
    run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
    print("Host compiler:", CXX)

    run([PYTHON, "-c", "import sglang, torch; "
         "print('sglang', sglang.__version__, '| torch', torch.__version__); "
         "assert torch.cuda.is_available(), 'CUDA unavailable'; "
         "print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))"],
        env=env)


    target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
    q = target_cfg.get("quantization_config", {})
    assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
    _, target_shards = indexed_shards(MODEL_DIR)
    print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
    DRAFT_VIEW = None
    if CFG["SPEC"]:
        assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
        DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)


    graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
    graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
    args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
        "--model-loader-extra-config", '{"enable_multithread_load":false}',
        "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
        "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
        "--dtype", "bfloat16", "--quantization", "auto-round",
        "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
        "--context-length", str(CFG["CTX"]), "--page-size", "64",
        "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
        "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
        "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
        "--cuda-graph-bs-decode", *map(str, graph_bs),
        "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
        "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
        "--mamba-backend", CFG["LINEAR_BACKEND"],
        "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
        "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
        "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
        "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
        "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
        "--default-chat-template-kwargs", '{"preserve_thinking":true}',
        "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
        "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
    if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
    if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
    tmpl = Path(MODEL_DIR, "chat_template.jinja")
    if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
    if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
    if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
    if CFG["SPEC"]:
        args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
            "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
            "--speculative-draft-model-path", str(DRAFT_VIEW),
            "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
            "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
            "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
            "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
        if CFG["FRSPEC"]:
            tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
            if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
            assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
            assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
            args += ["--speculative-token-map", str(tok)]


    return args, env


def _flash_healthy(timeout=2.0):
    try:
        with urllib.request.urlopen(f"{FLASH_URL}/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False


FLASH_PROC = None
FLASH_ARGS = FLASH_ENV = None


def launch_flash():
    """Start (or restart) the SGLang server process in the background; returns the Popen."""
    global FLASH_PROC
    with open(LOG, "ab", buffering=0) as logf:
        FLASH_PROC = subprocess.Popen(FLASH_ARGS, env=FLASH_ENV, stdout=logf, stderr=subprocess.STDOUT,
                                      start_new_session=True)
    print(f"SGLang pid {FLASH_PROC.pid} -> {LOG} (launched at {time.time() - NOTEBOOK_START_TIME:.0f}s)")
    return FLASH_PROC


def show_log_tail(n=6000):
    with contextlib.suppress(Exception):
        with open(LOG, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - n))
            print(f.read().decode(errors="replace"))


if _flash_healthy():
    print("Flash-Next server already running; reusing it")
else:
    precache(str(WHEELHOUSE_DIR))
    FLASH_ARGS, FLASH_ENV = _prepare_flashnext(str(FLASH_MODEL_DIR), str(FLASH_DRAFT_DIR or ""), str(WHEELHOUSE_DIR))
    threading.Thread(target=precache, args=(str(FLASH_MODEL_DIR), str(FLASH_DRAFT_DIR or FLASH_MODEL_DIR)),
                     kwargs={"threads": 3}, daemon=True).start()
    print("\n" + shlex.join(FLASH_ARGS) + "\n")
    launch_flash()


## Model backends

In [ ]:



import base64, contextlib, io, json, os, re, signal, threading, time, urllib.request
from PIL import Image


COACH_THINK = True
COACH_THINK_TOKENS = 1500


class FlashNextBackend:
    """Qwen3.8-Flash-Next (125B MoE, 6B active) behind SGLang's OpenAI-compatible API; thinking off, greedy."""
    name = 'flashnext'

    timing = dict(ACT_SECONDS=0.25, PLAN_SECONDS=2.5, PLAN_EVERY_S=15.0, PLAN_MIN_GAP_S=6.0,
                  ACT_MAX_TOKENS=56, PLAN_MAX_TOKENS=260)

    def __init__(self, url, model):
        self.url, self.model = url.rstrip('/'), model

    @staticmethod
    def _data_url(im):
        buf = io.BytesIO()
        im.save(buf, format='JPEG', quality=90)
        return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode()

    def generate(self, system, images, text, prefill, max_new, mode, budget=None):
        content = [{'type': 'image_url', 'image_url': {'url': self._data_url(im)}} for im in images]
        content.append({'type': 'text', 'text': text})
        think = COACH_THINK and mode == 'plan'
        body = {
            'model': self.model,
            'messages': [{'role': 'system', 'content': system}, {'role': 'user', 'content': content}],
            'temperature': 0.0, 'top_p': 1.0, 'top_k': 1,
            'max_tokens': int(max_new) + (COACH_THINK_TOKENS if think else 0),
            'chat_template_kwargs': {'enable_thinking': think},
            'skip_special_tokens': True,
        }
        if mode == 'line':
            body['stop'] = ['\n']
        req = urllib.request.Request(f'{self.url}/v1/chat/completions', data=json.dumps(body).encode(),
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=15 if mode == 'line' else 60) as r:
            msg = json.loads(r.read())['choices'][0]['message']

        out = (msg.get('content') or ('' if think else msg.get('reasoning_content')) or '').strip()
        out = re.sub(r'(?s)<think>.*?</think>', '', out).strip()
        if prefill and not out.lstrip().upper().startswith(prefill.rstrip(':').upper()):
            out = prefill + ' ' + out
        return out

    def warmup(self):
        img = Image.new('RGB', (640, 360), (40, 40, 40))
        self.generate('Reply with one word.', [img], 'Say ok.', '', 4, 'line')


class BackendManager:
    """Waits for the SGLang server; relaunches it (once per crash, up to 2 times) if it dies mid-run."""

    def __init__(self):
        self.flash = FlashNextBackend(FLASH_URL, SERVED_MODEL_NAME)
        self.backend, self.state, self.errors, self.relaunches = None, 'starting', 0, 0
        self._last_poll = 0.0
        self.timing = FlashNextBackend.timing

    def _relaunch(self, why):
        print(f'SGLang problem: {why}')
        show_log_tail()
        if self.relaunches >= 2 or FLASH_ARGS is None:
            return
        self.relaunches += 1
        with contextlib.suppress(Exception):
            if FLASH_PROC is not None and FLASH_PROC.poll() is None:
                os.killpg(FLASH_PROC.pid, signal.SIGKILL)
                FLASH_PROC.wait(timeout=60)
        time.sleep(5)
        launch_flash()
        self.backend, self.state = None, 'starting'

    def ready(self):
        if self.state == 'ready':
            return True
        now = time.time()
        if now - self._last_poll < 3.0:
            return False
        self._last_poll = now
        if _flash_healthy(1.5):
            try:
                self.flash.warmup()
                self.backend, self.state, self.errors = self.flash, 'ready', 0
                print(f'Flash-Next ready at notebook {now - NOTEBOOK_START_TIME:.0f}s')
                return True
            except Exception as e:
                print('Flash-Next warm-up failed:', repr(e)[:200])
        if FLASH_PROC is not None and FLASH_PROC.poll() is not None:
            self._relaunch(f'server exited with code {FLASH_PROC.returncode}')
        return False

    def wait_until(self, deadline_s, poll=2.0):
        while time.time() - NOTEBOOK_START_TIME < deadline_s:
            if self.ready():
                return True
            if FLASH_PROC is not None and FLASH_PROC.poll() is not None and self.relaunches >= 2:
                raise RuntimeError('SGLang keeps exiting; see /kaggle/working/serve.log')
            time.sleep(poll)
        return self.ready()

    def generate(self, *args, **kwargs):
        try:
            out = self.flash.generate(*args, **kwargs)
            self.errors = 0
            return out
        except Exception:
            self.errors += 1
            if self.errors >= 3 and not _flash_healthy(2.0):
                self.state, self.backend = 'starting', None
                if FLASH_PROC is not None and FLASH_PROC.poll() is not None:
                    self._relaunch(f'server exited with code {FLASH_PROC.returncode}')
            raise


if 'MANAGER' not in globals():
    MANAGER = BackendManager()


## The agent

## Background coach (from lightningv08)

The planner ("coach") runs in the background, as in lightningv08's notebook
[0.294 Qwen3.8-Flash-Next 125B background coach](https://www.kaggle.com/code/lightningv08/0-294-qwen3-8-flash-next-125b-background-coach).
Only one coach request is outstanding at a time; the actor keeps acting on every step while it runs.
A coach reply goes into the actor's notes; the actor still sees the current frame, so a stale menu hint meets a fresh screen.
A reply that lands more than 12 s of game time after its request started is discarded.
The coach's own ACT line is not executed.
Everything else in this notebook is gga-qwen-start unchanged.


In [ ]:

































import contextlib, dataclasses, io, re, time, hashlib
from collections import deque

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from gga.protocol import Action

try:
    from gga.protocol import NAMED_KEYS as _NAMED_KEYS
    NAMED_KEYS = {str(k) for k in _NAMED_KEYS}
except Exception:
    NAMED_KEYS = set()
NAMED_LOWER = {k.lower(): k for k in NAMED_KEYS}



def T(key):
    return MANAGER.timing[key]


TRAIL_HOLD_MS = 1600
MAX_PROGRAM_MS = 5000
IMG_SIDE = 1024
FROZEN_EPS = 0.004
BIG_CHANGE = 0.22
UNSTICK_AFTER = 6
GRID_OVERLAY = True

COMMANDS = '''Commands (coordinates are on a 0-999 grid over the screenshot: x 0=left..999=right, y 0=top..999=bottom; the faint magenta lines and labels mark every 100 units; MS = milliseconds):
  click X Y               left click
  rclick X Y              right click
  clicks X Y N            N fast left clicks on one spot (N up to 40, about 16 per second)
  move X Y [MS]           move the pointer there (hover / aim), then wait MS
  drag X1 Y1 X2 Y2        press at X1 Y1, drag to X2 Y2, release
  tap KEY [N]             press KEY once (or N times)
  hold KEYS [MS]          hold keys together for MS, join keys with + (example: hold ArrowRight+Space 400)
  hold KEYS+Mouse X Y [MS]  also hold the left mouse button with the pointer at X Y (aim and fire)
  wait MS                 do nothing
  type TEXT               type letters
Keys: single characters (w a s d r 1 2 ...) or ArrowLeft ArrowRight ArrowUp ArrowDown Space Enter Escape Shift Tab Backspace.
Chain commands with ";" to make a short program that runs in order, e.g.
  hold ArrowRight 500; hold ArrowRight+ArrowUp 300; hold ArrowRight
  clicks 480 520 15; click 850 200
  hold w+Mouse 610 340 600; hold d+Mouse 650 330
Your program runs to its end (up to 1.5 s) before you are asked again; a final hold without MS keeps going until your next reply, about half a second later. So in fast action games write short programs (0.3-0.8 s) that end with an open hold; write a whole combo (move then jump, click then buy) when you need one; never pad with long waits.
In grid, puzzle and turn-based games each tap is one move and holding a key usually moves only once: use tap KEY N there.'''

SYSTEM_ACT = f'''You are an expert video-game player controlling a real-time browser game from screenshots. The game keeps running while you think, so act decisively and keep inputs going. The score counts in-game progress (points, money, levels, distance, ...); doing nothing scores 0.

{COMMANDS}

How to play well:
- Follow the brief, the controls, and the NOTES from your coach.
- If a title, menu, pause, level-complete or game-over screen shows, get back into play: click its button (Play, Start, Retry, Continue, OK, Next) or press the key it names (often Enter or Space).
- Click exactly on the centre of the buttons and targets you see.
- If recent inputs caused no visible change, do something different.
- The magenta grid lines are a coordinate aid, not part of the game.
Reply with ONE line of commands and nothing else.'''

SYSTEM_PLAN = f'''You are an expert video-game player and coach for a real-time browser game you have never seen. You get screenshots from the last seconds (oldest first, the last one is NOW), the brief, your previous notes, and a log of recent inputs with how much the screen changed after each. Work out how this game is played and how to score more, then give the next inputs. The score counts in-game progress (points, money, levels, distance, ...); doing nothing scores 0.

{COMMANDS}

Tactics by kind of game:
- Clicker / idle / tycoon / management: click the main money source fast (clicks), buy upgrades and producers whenever affordable, keep buying.
- Platformer / runner: move toward the goal (usually right), jump over gaps, hazards and enemies, collect items.
- Shooter / arena / space: keep moving, aim at enemies and fire continuously, dodge bullets.
- Puzzle / board / match / line-drawing / grid movement: understand the rule, make deliberate moves that make progress, never repeat moves that did nothing; one tap = one move (tap KEY N), holding a key moves once.
- Racing / driving: hold accelerate, steer to stay on the track.
- Roguelike / adventure / exploration: explore, fight, pick up items, take exits or stairs to progress, avoid dying.
- Sports / physics / rhythm / typing: time inputs to what the screen shows.
- Title / menu / game-over screens: click Play/Start/Retry or press the key shown (Enter, Space).

Reflect like a coach reviewing a replay: if the score stopped rising, a life was lost or the game ended, say in LEARNED what caused it and how to avoid it; keep what worked, drop what did not. Ignore the magenta grid lines, they are not part of the game.

Think briefly and concretely (what is the goal, what do the inputs do, what went wrong), then reply in exactly this format, one short line each:
SCREEN: what is on screen now
SCORE: the score, money or level shown, or ?
LEARNED: what the inputs did and did not do so far
PLAN: the best concrete tactic for the next 30 seconds (where to click, which keys, what to avoid)
ACT: one line of commands to run now'''


_ALIAS = {
    'space': 'Space', 'spacebar': 'Space', 'enter': 'Enter', 'return': 'Enter', 'esc': 'Escape',
    'escape': 'Escape', 'tab': 'Tab', 'shift': 'Shift', 'shiftleft': 'Shift', 'lshift': 'Shift',
    'ctrl': 'Control', 'control': 'Control', 'controlleft': 'Control', 'alt': 'Alt', 'backspace': 'Backspace',
    'delete': 'Delete', 'del': 'Delete', 'up': 'ArrowUp', 'arrowup': 'ArrowUp', 'uparrow': 'ArrowUp',
    'down': 'ArrowDown', 'arrowdown': 'ArrowDown', 'downarrow': 'ArrowDown', 'left': 'ArrowLeft',
    'arrowleft': 'ArrowLeft', 'leftarrow': 'ArrowLeft', 'right': 'ArrowRight', 'arrowright': 'ArrowRight',
    'rightarrow': 'ArrowRight', 'pageup': 'PageUp', 'pagedown': 'PageDown', 'home': 'Home', 'end': 'End',
    'insert': 'Insert', '↑': 'ArrowUp', '↓': 'ArrowDown', '←': 'ArrowLeft', '→': 'ArrowRight',
}
_LMB = {'mouse', 'lmb', 'mouseleft', 'leftmouse', 'leftclick', 'mouse1', 'leftbutton', 'lclick'}
_RMB = {'rmb', 'mouseright', 'rightmouse', 'rightclick', 'mouse2', 'rightbutton'}
_NUM = re.compile(r'^-?\d+(?:\.\d+)?$')


def norm_key(tok):
    t = tok.strip().strip('"\'`.,:[]{}()')
    if not t:
        return None
    low = t.lower()
    if low in _ALIAS:
        k = _ALIAS[low]
    elif len(t) == 1:
        k = low if t.isalpha() else t
    elif low in NAMED_LOWER:
        k = NAMED_LOWER[low]
    elif re.fullmatch(r'f\d{1,2}', low):
        k = low.upper()
    elif re.fullmatch(r'key[a-z]', low):
        k = low[3]
    elif re.fullmatch(r'digit\d', low):
        k = low[5]
    else:
        return None
    if NAMED_KEYS and len(k) > 1 and k not in NAMED_KEYS:
        return NAMED_LOWER.get(k.lower())
    return k


def _clamp(v, lo, hi):
    return max(lo, min(hi, v))



_VERBS = {'click', 'lclick', 'rclick', 'rightclick', 'dblclick', 'doubleclick', 'clicks', 'move', 'hover', 'aim',
          'drag', 'swipe', 'tap', 'press', 'key', 'hold', 'keys', 'wait', 'sleep', 'idle', 'type', 'skip', 'noop'}


def _pick_line(text):
    text = text.replace('`', ' ').replace('*', ' ')
    m = re.search(r'ACT\s*:\s*([^\n]*)', text)
    if m and m.group(1).strip():
        return m.group(1)
    for line in text.splitlines():
        words = line.replace(';', ' ').split()
        if words and words[0].lower().rstrip(':') in _VERBS:
            return line
    return text.splitlines()[0] if text.strip() else ''


def parse_program(text):
    ops = []
    for part in re.split(r'[;|]', _pick_line(text)):
        toks = re.sub(r'[,()=]', ' ', part).split()
        if not toks:
            continue
        verb = toks[0].lower().rstrip(':')
        args = toks[1:]
        nums = [float(a) for a in args if _NUM.match(a)]
        words = [a for a in args if not _NUM.match(a)]
        xy = lambda i=0: (_clamp(int(nums[i]), 0, 999), _clamp(int(nums[i + 1]), 0, 999))
        try:
            if verb in ('click', 'lclick') and len(nums) >= 2:
                ops.append({'op': 'click', 'xy': xy(), 'b': 'left'})
            elif verb in ('rclick', 'rightclick') and len(nums) >= 2:
                ops.append({'op': 'click', 'xy': xy(), 'b': 'right'})
            elif verb in ('dblclick', 'doubleclick') and len(nums) >= 2:
                ops.append({'op': 'clicks', 'xy': xy(), 'n': 2})
            elif verb == 'clicks' and len(nums) >= 2:
                n = int(nums[2]) if len(nums) >= 3 else 10
                ops.append({'op': 'clicks', 'xy': xy(), 'n': _clamp(n, 1, 40)})
            elif verb in ('move', 'hover', 'aim') and len(nums) >= 2:
                ms = int(nums[2]) if len(nums) >= 3 else 0
                ops.append({'op': 'move', 'xy': xy(), 'ms': _clamp(ms, 0, 3000)})
            elif verb in ('drag', 'swipe') and len(nums) >= 4:
                ops.append({'op': 'drag', 'xy': xy(0), 'xy2': xy(2)})
            elif verb in ('tap', 'press', 'key') and words:
                keys = [norm_key(k) for w in words for k in w.split('+')]
                keys = [k for k in keys if k]
                n = _clamp(int(nums[0]), 1, 20) if nums else 1
                if len(keys) == 1:
                    ops.append({'op': 'tap', 'key': keys[0], 'n': n})
                elif keys:
                    ops.append({'op': 'hold', 'keys': keys, 'buttons': [], 'xy': None, 'ms': 120})
            elif verb in ('hold', 'keys') or (verb not in _VERBS and norm_key(toks[0]) and len(toks[0]) > 1):
                if verb not in ('hold', 'keys'):
                    words = [toks[0]] + words
                keys, buttons = [], []
                for w in words:
                    for k in w.split('+'):
                        lk = k.lower().strip()
                        if lk in _LMB:
                            buttons.append('left')
                        elif lk in _RMB:
                            buttons.append('right')
                        else:
                            nk = norm_key(k)
                            if nk and nk not in keys:
                                keys.append(nk)
                pos, ms = None, None
                if len(nums) >= 3:
                    pos, ms = xy(), int(nums[2])
                elif len(nums) == 2:
                    pos = xy()
                elif len(nums) == 1:
                    ms = int(nums[0])
                if keys or buttons:
                    ops.append({'op': 'hold', 'keys': keys, 'buttons': sorted(set(buttons)), 'xy': pos,
                                'ms': None if ms is None else _clamp(ms, 0, 3000)})
            elif verb in ('wait', 'sleep', 'idle'):
                ops.append({'op': 'wait', 'ms': _clamp(int(nums[0]) if nums else 300, 0, 3000)})
            elif verb == 'type':
                txt = part.split(None, 1)[1].strip().strip('"\'') if len(part.split(None, 1)) > 1 else ''
                if txt:
                    ops.append({'op': 'type', 'text': txt[:30]})
            elif verb in ('skip', 'noop'):
                ops.append({'op': 'wait', 'ms': 300})
        except (ValueError, IndexError):
            continue
        if len(ops) >= 12:
            break
    return ops


def fmt_program(ops):
    out = []
    for o in ops:
        k = o['op']
        if k == 'click':
            out.append(f"{'click' if o['b'] == 'left' else 'rclick'} {o['xy'][0]} {o['xy'][1]}")
        elif k == 'clicks':
            out.append(f"clicks {o['xy'][0]} {o['xy'][1]} {o['n']}")
        elif k == 'move':
            out.append(f"move {o['xy'][0]} {o['xy'][1]}" + (f" {o['ms']}" if o['ms'] else ''))
        elif k == 'drag':
            out.append(f"drag {o['xy'][0]} {o['xy'][1]} {o['xy2'][0]} {o['xy2'][1]}")
        elif k == 'tap':
            out.append(f"tap {o['key']}" + (f" {o['n']}" if o['n'] > 1 else ''))
        elif k == 'hold':
            ks = '+'.join(o['keys'] + (['Mouse'] if 'left' in o['buttons'] else []))
            s = f"hold {ks}"
            if o['xy']:
                s += f" {o['xy'][0]} {o['xy'][1]}"
            if o['ms'] is not None:
                s += f" {o['ms']}"
            out.append(s)
        elif k == 'wait':
            out.append(f"wait {o['ms']}")
        elif k == 'type':
            out.append(f"type {o['text']}")
    return '; '.join(out)



def _as_dict(t):
    if isinstance(t, dict):
        return t
    if hasattr(t, 'model_dump'):
        return t.model_dump()
    if dataclasses.is_dataclass(t):
        return dataclasses.asdict(t)
    return dict(getattr(t, '__dict__', {}))


class ToolTable:
    def __init__(self, tools):
        self.schemas = {}
        for t in tools or []:
            d = _as_dict(t)
            name = d.get('name') or getattr(t, 'name', None)
            schema = d.get('parameters') or d.get('input_schema') or d.get('schema') or d.get('params') or {}
            if name:
                self.schemas[str(name)] = schema if isinstance(schema, dict) else {}
        self.has = lambda n: (not self.schemas) or n in self.schemas
        seq = self.schemas.get('sequence', {})
        steps = (seq.get('properties') or {}).get('steps') or {}
        items = steps.get('items') or {}
        self.step_fields = set((items.get('properties') or {}).keys())
        self.max_steps = int(steps.get('maxItems') or 200)
        dur = (items.get('properties') or {}).get('duration_ms') or {}
        self.max_dur = int(dur.get('maximum') or 10000)

    def props(self, name):
        return set(((self.schemas.get(name) or {}).get('properties') or {}).keys())



class Compiler:
    def __init__(self, W, H, table, pointer, box=None):
        self.W, self.H, self.t = W, H, table
        self.box = box or (0, 0, W, H)
        self.px, self.py = pointer if pointer else (W // 2, H // 2)
        self.committed_ms = 0

    def pix(self, xy):
        x0, y0, x1, y1 = self.box
        return (int(round(x0 + xy[0] * (x1 - x0 - 1) / 999)), int(round(y0 + xy[1] * (y1 - y0 - 1) / 999)))

    def single(self, o):
        """One op as a plain tool, when that is enough (fewer moving parts, never refused for length)."""
        t = self.t
        if o['op'] == 'click' and t.has('click'):
            x, y = self.pix(o['xy'])
            p = {'x': x, 'y': y}
            if o['b'] != 'left':
                if 'button' not in t.props('click'):
                    return None
                p['button'] = o['b']
            self.px, self.py = x, y
            return Action('click', p)
        if o['op'] == 'tap' and o['n'] == 1 and t.has('press_key'):
            return Action('press_key', {'key': o['key']})
        if o['op'] == 'move' and not o['ms'] and t.has('move'):
            x, y = self.pix(o['xy'])
            self.px, self.py = x, y
            return Action('move', {'x': x, 'y': y})
        if o['op'] == 'drag' and t.has('drag'):
            props = t.props('drag')
            (x1, y1), (x2, y2) = self.pix(o['xy']), self.pix(o['xy2'])
            if {'x1', 'y1', 'x2', 'y2'} <= props:
                self.px, self.py = x2, y2
                return Action('drag', {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2})
        return None

    def sequence(self, ops):
        steps, total, open_ms = [], 0, 0
        cap = max(4, self.t.max_steps) - 1

        def add(keys, buttons, ms, pos=None):
            nonlocal total
            if len(steps) >= cap:
                return
            if pos is not None:
                self.px, self.py = pos
            ms = int(_clamp(ms, 0, self.t.max_dur))
            steps.append({'keys': list(keys), 'buttons': list(buttons), 'x': self.px, 'y': self.py,
                          'duration_ms': ms})
            total += ms

        for i, o in enumerate(ops):
            if total >= MAX_PROGRAM_MS or len(steps) >= cap:
                break
            last = i == len(ops) - 1
            k = o['op']
            if k == 'click':
                add([], [], 20, self.pix(o['xy'])); add([], [o['b']], 50); add([], [], 40)
            elif k == 'clicks':
                add([], [], 20, self.pix(o['xy']))
                for _ in range(o['n']):
                    add([], ['left'], 30); add([], [], 30)
            elif k == 'move':
                add([], [], max(40, o['ms']), self.pix(o['xy']))
            elif k == 'drag':
                (x1, y1), (x2, y2) = self.pix(o['xy']), self.pix(o['xy2'])
                add([], [], 30, (x1, y1)); add([], ['left'], 60, (x1, y1))
                for j in range(1, 7):
                    add([], ['left'], 30, (x1 + (x2 - x1) * j // 6, y1 + (y2 - y1) * j // 6))
                add([], ['left'], 60, (x2, y2)); add([], [], 30)
            elif k == 'tap':
                for _ in range(o['n']):
                    add([o['key']], [], 70); add([], [], 40)
            elif k == 'hold':
                ms = o['ms'] if o['ms'] is not None else (TRAIL_HOLD_MS if last else 300)
                if last and o['ms'] is None:
                    open_ms = ms
                add(o['keys'], o['buttons'], ms, self.pix(o['xy']) if o['xy'] else None)
            elif k == 'wait':
                add([], [], o['ms'])
            elif k == 'type':
                for ch in o['text']:
                    key = 'Space' if ch == ' ' else (ch.lower() if ch.isalpha() else ch)
                    add([key], [], 50); add([], [], 30)
        steps.append({'keys': [], 'buttons': [], 'x': self.px, 'y': self.py, 'duration_ms': 0})
        self.committed_ms = max(0, total - open_ms)
        if self.t.step_fields:
            steps = [{k: v for k, v in s.items() if k in self.t.step_fields} for s in steps]
        return Action('sequence', {'steps': steps})

    def fallback(self, o):
        """Without `sequence`: the first op as the closest single tool."""
        a = self.single(o)
        if a is not None:
            return a
        t = self.t
        if o['op'] in ('click', 'clicks') and t.has('click'):
            x, y = self.pix(o['xy'])
            self.px, self.py = x, y
            return Action('click', {'x': x, 'y': y})
        if o['op'] == 'tap' and t.has('press_key'):
            return Action('press_key', {'key': o['key']})
        if o['op'] == 'hold' and o['keys']:
            props = t.props('hold_keys')
            if 'hold_keys' in t.schemas and 'keys' in props:
                p = {'keys': o['keys']}
                for name in ('duration_ms', 'ms', 'duration'):
                    if name in props:
                        p[name] = o['ms'] if o['ms'] is not None else TRAIL_HOLD_MS
                return Action('hold_keys', p)
            if t.has('press_key'):
                return Action('press_key', {'key': o['keys'][0]})
        if o['op'] == 'hold' and o['xy'] and t.has('click'):
            x, y = self.pix(o['xy'])
            return Action('click', {'x': x, 'y': y})
        return Action('skip')

    def compile(self, ops, use_sequence):
        if not ops:
            return Action('skip')
        if len(ops) == 1:
            a = self.single(ops[0])
            if a is not None:
                return a
        if use_sequence and self.t.has('sequence'):
            return self.sequence(ops)
        return self.fallback(ops[0])



def _font(size):
    try:
        return ImageFont.load_default(size=size)
    except Exception:
        return ImageFont.load_default()


def grid_overlay(img):
    """Faint lines every 100 units of the 0-999 action grid with edge labels. The labels sit on the model's own
    normalised scale, so they anchor its coordinates without changing the mapping."""
    im = img.convert('RGBA')
    w, h = im.size
    layer = Image.new('RGBA', im.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    fs = max(10, min(w, h) // 45)
    font = _font(fs)
    for v in range(100, 1000, 100):
        x = round(v * (w - 1) / 999)
        y = round(v * (h - 1) / 999)
        d.line([(x, 0), (x, h)], fill=(255, 0, 255, 46), width=1)
        d.line([(0, y), (w, y)], fill=(255, 0, 255, 46), width=1)
        for (tx, ty) in ((x + 2, 1), (1, y + 1)):
            d.rectangle([tx - 1, ty - 1, tx + 3 * fs * 0.62, ty + fs + 1], fill=(0, 0, 0, 120))
            d.text((tx, ty), str(v), fill=(255, 255, 255, 220), font=font)
    return Image.alpha_composite(im, layer).convert('RGB')



def _small(img):
    return np.asarray(img.convert('L').resize((96, 54), Image.BILINEAR), dtype=np.float32) / 255.0


def _screen_key(small):
    q = (small[::6, ::6] * 7).astype(np.uint8)
    return hashlib.md5(q.tobytes()).hexdigest()[:12]


def _jpeg_of(item):
    if isinstance(item, (bytes, bytearray)):
        return bytes(item)
    if isinstance(item, (tuple, list)):
        for v in item:
            if isinstance(v, (bytes, bytearray)):
                return bytes(v)
    for name in ('image', 'jpeg', 'data'):
        v = getattr(item, name, None)
        if isinstance(v, (bytes, bytearray)):
            return bytes(v)
    return None


def _refusal(last):
    """Error text if the previous reply was refused, else ''."""
    if last is None:
        return ''
    d = last if isinstance(last, dict) else _as_dict(last)
    if not d:
        s = str(last)
        return s[:160] if re.search(r'refus|invalid|error', s, re.I) else ''
    ok = d.get('ok', True)
    err = d.get('error') or d.get('reason') or ''
    if ok is False or err:
        return str(err or 'refused')[:160]
    return ''



UNSTICK = [
    'click 500 500', 'tap Enter', 'tap Space', 'click 500 620', 'click 500 760', 'tap r',
    'click 500 380', 'hold ArrowRight 600; tap ArrowUp', 'click 500 880', 'tap Escape',
]


from concurrent.futures import ThreadPoolExecutor

COACH_STALE_S = 16.0


class BackgroundCoach:
    """One outstanding planner request across games (after lightningv08); the second server slot stays for the actor."""

    def __init__(self, client):
        self.client = client
        self.pool = ThreadPoolExecutor(max_workers=1)
        self.current = None

    def available(self):
        return self.current is None or self.current.done()

    def start(self, request):
        self.current = self.pool.submit(self.client.generate, *request)
        return self.current


BACKGROUND_COACH = BackgroundCoach(FlashNextBackend(FLASH_URL, SERVED_MODEL_NAME))



COMMIT_MAX_S = 1.5
CROP = True
CROP_TOL = 20
CROP_MIN_CUT = 0.15
CROP_MIN_SIDE = 0.2
CROP_MARGIN = 0.03
CHANGE_CELL = 0.06
COACH_ACT_MAX_AGE_S = 12.0
UPSCALE_TO = 448


def content_box(frame):
    """Bounding box (frame px) of everything that differs from a flat page colour around the frame, or None
    when the frame's border is not one flat colour (the game reaches the edge of the page)."""
    fw, fh = frame.size
    k = 4 if min(fw, fh) >= 400 else 1
    a = np.asarray(frame.reduce(k) if k > 1 else frame, dtype=np.int16)
    h, w, _ = a.shape
    ring = np.concatenate([a[:2].reshape(-1, 3), a[-2:].reshape(-1, 3),
                           a[:, :2].reshape(-1, 3), a[:, -2:].reshape(-1, 3)])
    bg = np.median(ring, axis=0)
    if (np.abs(ring - bg).max(1) > CROP_TOL).mean() > 0.03:
        return None
    mask = np.abs(a - bg).max(2) > CROP_TOL
    rows = np.flatnonzero(mask.sum(1) >= 2)
    cols = np.flatnonzero(mask.sum(0) >= 2)
    if rows.size == 0 or cols.size == 0:
        return (fw * 0.5, fh * 0.5, fw * 0.5, fh * 0.5)
    sx, sy = fw / w, fh / h
    return (cols[0] * sx, rows[0] * sy, (cols[-1] + 1) * sx, (rows[-1] + 1) * sy)


def _change_where(small, prev):
    """Where two 96x54 thumbnails differ: (x0, y0, x1, y1) as shares of the frame, 'all', or None."""
    m = np.abs(small - prev) > CHANGE_CELL
    n = int(m.sum())
    if n < 2:
        return None
    if n > 0.5 * m.size:
        return 'all'
    ys, xs = np.nonzero(m)
    h, w = m.shape
    return (xs.min() / w, ys.min() / h, (xs.max() + 1) / w, (ys.max() + 1) / h)


def _union_where(a, b):
    if a is None:
        return b
    if b is None:
        return a
    if a == 'all' or b == 'all':
        return 'all'
    return (min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3]))


class QwenAgent:
    def __init__(self):
        self.last_elapsed = float('inf')
        self.desc = None
        self.table, self._tools_id = ToolTable([]), None
        self._new_game()


    def _new_game(self):
        self.steps = 0
        self.notes = ''
        self.plan_t = None
        self.pending_plan = None
        self.coach_view_box = None
        self.last_scene_change = float('-inf')
        self.log = deque(maxlen=8)
        self.frames = deque(maxlen=48)
        self.prev_small = None
        self.frozen = 0
        self.scores = deque(maxlen=6)
        self.pointer = None
        self.use_sequence = True
        self.seq_refusals = 0
        self.last_tool = None
        self.noop = {}
        self.unstick_i = 0
        self.last_action = 'none'
        self.thought_s = 0.0
        self.mode = ''
        self.run_entry = None
        self.next_entry = None
        self.commit_until = -1.0
        self.lat_est = 0.4
        self.win_active = 0.0
        self.union, self.full = None, False
        self.box = self.fbox = None
        self.view_box = None
        self.fsize = None
        self.coach_act = None


    def step(self, obs, tools):
        t0 = time.time()
        pad = T('ACT_SECONDS')
        try:
            action, pad = self._step(obs, tools)
        except Exception as e:
            print('agent error:', repr(e)[:300])
            action = Action('skip')
        self.thought_s = time.time() - t0
        time.sleep(max(0.0, pad - self.thought_s))
        return action


    def _step(self, obs, tools):
        el = float(getattr(obs, 'elapsed_s', 0.0) or 0.0)
        info = getattr(obs, 'info', None) or {}
        desc = str(info.get('description', ''))
        if el < self.last_elapsed - 0.5 or (self.desc is not None and desc != self.desc):
            self._new_game()
        self.desc = desc
        self.last_elapsed = el
        if id(tools) != self._tools_id:
            self.table, self._tools_id = ToolTable(tools), id(tools)

        frame = Image.open(io.BytesIO(obs.image)).convert('RGB')
        W = int(info.get('width') or frame.size[0])
        H = int(info.get('height') or frame.size[1])


        small = _small(frame)
        delta = 1.0 if self.prev_small is None else float(np.abs(small - self.prev_small).mean())
        motion = 0.0
        recent = list(getattr(obs, 'frames', None) or [])
        if len(recent) >= 2:
            a, b = _jpeg_of(recent[0]), _jpeg_of(recent[-1])
            if a and b:
                motion = float(np.abs(_small(Image.open(io.BytesIO(b))) - _small(Image.open(io.BytesIO(a)))).mean())
        active = max(delta, motion) if self.prev_small is not None else 1.0


        if self.run_entry is not None and self.prev_small is not None:
            e = self.run_entry
            e['change'] = max(e['change'] or 0.0, delta)
            e['active'] = max(e['active'], active)
            e['where'] = _union_where(e.get('where'), _change_where(small, self.prev_small))
        if self.next_entry is not None:
            self._finish(self.run_entry)
            self.run_entry, self.next_entry = self.next_entry, None
            self.lat_est = _clamp(0.7 * self.lat_est + 0.3 * (el - self.run_entry['t']), 0.2, 1.5)
            self.commit_until = el + self.run_entry['commit']
        self.win_active = max(self.win_active, active)
        self.prev_small = small
        skey = _screen_key(small)

        refused = _refusal(getattr(obs, 'last_action', None))
        if refused and self.last_tool == 'sequence':
            self.seq_refusals += 1
            if self.seq_refusals >= 3:
                self.use_sequence = False
        elif self.last_tool == 'sequence':
            self.seq_refusals = 0

        self._update_crop(frame, W, H)
        if delta > BIG_CHANGE:
            self.last_scene_change = el
            self.coach_act = None
        self.fsize = frame.size
        self.frames.append((el, obs.image))
        self.steps += 1

        if not MANAGER.ready():
            self.mode = 'no-model'
            ops = parse_program(UNSTICK[self.unstick_i % len(UNSTICK)])
            self.unstick_i += 1
            return self._send(ops, W, H, el, None), 1.0

        since = None if self.plan_t is None else el - self.plan_t
        need_plan = (since is None or since >= T('PLAN_EVERY_S')
                     or (self.frozen >= 3 and since >= T('PLAN_MIN_GAP_S'))
                     or (delta > BIG_CHANGE and since >= T('PLAN_MIN_GAP_S'))
                     or (self._looping() and since >= T('PLAN_MIN_GAP_S')))

        if self.pending_plan is not None and self.pending_plan.done():
            try:
                advice = self.pending_plan.result()
            except Exception as e:
                print('coach error:', repr(e)[:200])
                advice = ''
            self.pending_plan = None
            if advice and el - self.plan_t <= COACH_STALE_S:
                self._read_advice(advice, self.plan_t)
        if need_plan and self.pending_plan is None and BACKGROUND_COACH.available():
            self.pending_plan = BACKGROUND_COACH.start(self._plan_request(info, el, refused))
            self.plan_t = el
            self.coach_view_box = self.fbox



        if (self.run_entry is not None and el + self.lat_est < self.commit_until - 0.05
                and delta <= BIG_CHANGE and not refused):
            self.mode = 'commit'
            return Action('skip'), 0.0

        self.frozen = self.frozen + 1 if self.win_active < FROZEN_EPS else 0
        self.win_active = 0.0

        if self.frozen >= UNSTICK_AFTER and self.frozen % 2 == 0:
            self.mode = 'unstick'
            text = UNSTICK[self.unstick_i % len(UNSTICK)]
            self.unstick_i += 1
        elif self._can_play_coach(el, refused):

            self.mode = 'coach' if self.frozen >= 3 else 'coach-loop'
            text, self.coach_act = self.coach_act[0], None
        else:
            self.mode = 'act'
            img = self._view(frame)
            try:
                text = MANAGER.generate(SYSTEM_ACT, [img], self._act_prompt(info, el, refused, skey), '',
                                        T('ACT_MAX_TOKENS'), 'line')
            except Exception as e:
                print('actor error:', repr(e)[:160])
                text = ''

        ops = parse_program(text)
        if self.mode == 'act' and not ops and self.frozen < 3:
            good = self._last_good()
            if good:
                ops, self.mode = good, 'repeat'
        return self._send(ops, W, H, el, skey), T('ACT_SECONDS')

    def _can_play_coach(self, el, refused):
        if self.coach_act is None or refused:
            return False
        text, asked_at, view_box = self.coach_act
        if (asked_at < self.last_scene_change or el - asked_at > COACH_ACT_MAX_AGE_S
                or view_box != self.fbox):
            return False
        if self.frozen >= 3:
            return bool(parse_program(text))
        if not self._looping():
            return False
        ops = parse_program(text)
        return bool(ops) and fmt_program(ops) != self.log[-1]['cmd']

    def _send(self, ops, W, H, el, skey):
        comp = Compiler(W, H, self.table, self.pointer, self.box)
        action = comp.compile(ops, self.use_sequence)
        self.pointer = (comp.px, comp.py)
        self.last_tool = action.tool
        cmd = fmt_program(ops) or 'skip'
        self.last_action = f'[{self.mode}] {cmd}'
        entry = {'t': el, 'cmd': cmd, 'change': None, 'active': 0.0, 'key': skey if ops else None,
                 'ops': ops if self.mode in ('act', 'repeat', 'coach', 'coach-loop') and ops else None, 'where': None,
                 'commit': min(comp.committed_ms / 1000.0, COMMIT_MAX_S) if action.tool == 'sequence' else 0.0}
        self.log.append(entry)
        self.next_entry = entry
        return action

    def _finish(self, e):
        """A program was replaced: if the screen never moved while it ran, remember it did nothing here."""
        if e is not None and e['key'] and e['change'] is not None and e['active'] < FROZEN_EPS:
            self.noop.setdefault(e['key'], set()).add(e['cmd'])

    def _last_good(self):
        for e in reversed(list(self.log)[-5:]):
            if e['ops'] and e['change'] is not None and e['active'] >= FROZEN_EPS:
                return e['ops']
        return None


    def _update_crop(self, frame, W, H):
        if not CROP or self.full:
            self.box = self.fbox = None
            return
        b = content_box(frame)
        if b is None:
            self.full, self.box, self.fbox = True, None, None
            return
        u = self.union
        self.union = b if u is None else (min(u[0], b[0]), min(u[1], b[1]), max(u[2], b[2]), max(u[3], b[3]))
        fw, fh = frame.size
        x0, y0, x1, y1 = self.union
        x0, y0 = max(0.0, x0 - CROP_MARGIN * fw), max(0.0, y0 - CROP_MARGIN * fh)
        x1, y1 = min(float(fw), x1 + CROP_MARGIN * fw), min(float(fh), y1 + CROP_MARGIN * fh)
        if (x1 - x0 < CROP_MIN_SIDE * fw or y1 - y0 < CROP_MIN_SIDE * fh
                or (x1 - x0) * (y1 - y0) > (1 - CROP_MIN_CUT) * fw * fh):
            self.box = self.fbox = None
            return
        self.fbox = tuple(int(round(v)) for v in (x0, y0, x1, y1))
        sx, sy = W / fw, H / fh
        self.box = (int(round(x0 * sx)), int(round(y0 * sy)), int(round(x1 * sx)), int(round(y1 * sy)))

    def _view(self, frame):
        """What the model sees: the game area (or the whole frame), at least UPSCALE_TO px, grid on top."""
        im = frame.crop(self.fbox) if self.fbox else frame.copy()
        side = max(im.size)
        if side < UPSCALE_TO:
            k = min(3, -(-UPSCALE_TO // side))
            im = im.resize((im.size[0] * k, im.size[1] * k), Image.NEAREST)
        im.thumbnail((IMG_SIDE, IMG_SIDE))
        self.view_box = self.box
        return grid_overlay(im) if GRID_OVERLAY else im

    def _where_text(self, e):
        """' at x A-B y C-D' on the model's 0-999 grid of the current view, or '' when unknown / everywhere."""
        w = e.get('where')
        if not w or e['change'] is None or e['change'] < FROZEN_EPS:
            return ''
        if w == 'all':
            return ' (most of the screen)'
        x0, y0, x1, y1 = w
        if self.fbox and self.fsize:
            fw, fh = self.fsize
            bx0, by0, bx1, by1 = self.fbox
            conv = lambda v, f, a, b: (v * f - a) / max(1, b - a)
            x0, x1 = conv(x0, fw, bx0, bx1), conv(x1, fw, bx0, bx1)
            y0, y1 = conv(y0, fh, by0, by1), conv(y1, fh, by0, by1)
        x0, x1, y0, y1 = (int(_clamp(v * 999, 0, 999)) for v in (x0, x1, y0, y1))
        if (x1 - x0) * (y1 - y0) > 0.5 * 999 * 999:
            return ' (most of the screen)'
        return f' at x {x0}-{x1} y {y0}-{y1}'

    def _looping(self):
        if len(self.log) < 4:
            return False
        last = list(self.log)[-4:]
        same = len({e['cmd'] for e in last}) == 1
        changes = [e['change'] for e in last if e['change'] is not None]
        return same and bool(changes) and max(changes) < 0.02


    def _brief(self, info):
        parts = []
        for key in ('title', 'genre'):
            if info.get(key):
                parts.append(f'{key.capitalize()}: {info[key]}')
        parts.append(f"Brief: {info.get('description', '')}")
        parts.append(f"Controls: {info.get('controls', '')}")
        return '\n'.join(parts)

    def _log_text(self, n):
        rows = []
        for e in list(self.log)[-n:]:
            ch = 'not seen yet' if e['change'] is None else f"{100 * e['change']:.1f}%" + self._where_text(e)
            rows.append(f"  t={e['t']:.0f}s: {e['cmd'][:90]} -> screen change while it ran: {ch}")
        return '\n'.join(rows) if rows else '  (none yet)'

    def _warnings(self, refused, skey):
        w = []
        if refused:
            w.append(f'The game refused your previous input: {refused}')
        if self.frozen >= 3:
            w.append(f'The screen has not changed for {self.frozen} replies: your inputs are not working. '
                     'Try something different (click a visible button, Enter, Space, other keys).')
        if self._looping():
            w.append('You repeated the same input 4 times with almost no effect. Change it.')
        dead = sorted(self.noop.get(skey, ()))[:5]
        if dead:
            w.append('On this screen these inputs did nothing: ' + ' | '.join(d[:60] for d in dead))
        return '\n'.join(w)

    def _time(self, info, el):
        rem = info.get('remaining_s')
        return f'Game time: {el:.0f} s played' + (f', {float(rem):.0f} s left.' if rem is not None else '.')

    def _act_prompt(self, info, el, refused, skey):
        text = [self._brief(info), self._time(info, el)]
        if self.notes:
            text.append('NOTES from your coach:\n' + self.notes)
        text.append('Recent inputs (oldest first):\n' + self._log_text(4))
        w = self._warnings(refused, skey)
        if w:
            text.append('WARNING: ' + w)
        text.append('The screenshot is NOW. Your commands:')
        return '\n'.join(text)

    def _plan_request(self, info, el, refused):
        imgs = self._plan_images(el)
        text = [self._brief(info), self._time(info, el)]
        if self.notes:
            text.append('Your previous notes:\n' + self.notes)
        if self.scores:
            text.append('Score readings so far: ' + ', '.join(f'{t:.0f}s: {s}' for t, s in self.scores))
        text.append('Recent inputs (oldest first):\n' + self._log_text(8))
        w = self._warnings(refused, _screen_key(self.prev_small))
        if w:
            text.append('WARNING: ' + w)
        ages = ', '.join(f'{el - t:.0f} s ago' for t, _ in imgs[:-1])
        text.append(f'Screenshots: {ages + ", " if ages else ""}and NOW (last).')
        return (SYSTEM_PLAN, [im for _, im in imgs], '\n'.join(text), 'SCREEN:', T('PLAN_MAX_TOKENS'), 'plan')

    def _read_advice(self, out, el):
        fields = {}
        for line in out.splitlines():
            m = re.match(r'\s*(SCREEN|SCORE|LEARNED|PLAN|ACT)\s*:\s*(.*)', line)
            if m:
                fields[m.group(1)] = m.group(2).strip()
        notes = [f'{k}: {fields[k][:220]}' for k in ('SCREEN', 'LEARNED', 'PLAN') if fields.get(k)]
        if notes:
            self.notes = '\n'.join(notes)
        if fields.get('ACT'):
            self.coach_act = (fields['ACT'], el, self.coach_view_box)
        sc = fields.get('SCORE', '').strip()
        if sc and sc != '?':
            self.scores.append((el, sc[:24]))

    def _plan_images(self, el):
        """Frames ~8 s and ~3 s ago and now, all cut to the CURRENT game area, so every image (and the
        coordinates in the coach's notes) share one frame of reference with the actor's screenshot."""
        frames = list(self.frames)
        now = frames[-1]
        chosen = []
        for age in (8.0, 3.0):
            older = [f for f in frames[:-1] if f[0] <= el - age + 0.6]
            if older and all(older[-1][0] != c[0] for c in chosen):
                chosen.append(older[-1])
        return [(t, self._view(Image.open(io.BytesIO(j)).convert('RGB'))) for t, j in chosen + [now]]


In [ ]:
Path('/kaggle/working/o1').mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/kaggle/working/o1')


In [ ]:
%%writefile /kaggle/working/o1/o1_perception.py
"""Generic perception from pixels, with no model: one Scene per frame.

A Scene is the frame reduced to GRID cells. From it, code answers four questions:
  - did the screen change (colour_signature / signature_distance: scrolling keeps it, a new screen changes it);
  - is this a known menu screen (ink_match: do the menu's non-background cells still show);
  - where are the things that stand out (salient: rare-colour blobs);
  - where is a named colour, and which way is it from the player (colour cells, direction_keys).
It knows no game and no object, only colours and positions.
"""
import colorsys
import io
from collections import Counter

from PIL import Image


GRID = (96, 60)
RARE_SHARE = 0.04
CENTRE_HOLE = 0.06
INK_SHARE = 0.10
MIN_INK = 0.01
PLAYER_HOLE = 0.06
TOUCH = 0.025
AXIS_SHARE = 0.4

COLOURS = ("red", "orange", "yellow", "green", "cyan", "blue", "purple", "pink", "white", "black")

_HUES = ((0.042, "red"), (0.125, "orange"), (0.19, "yellow"), (0.46, "green"), (0.53, "cyan"), (0.70, "blue"),
         (0.80, "purple"), (0.93, "pink"), (1.01, "red"))


def colour_name(r, g, b):
    h, s, v = colorsys.rgb_to_hsv(r / 255, g / 255, b / 255)
    if v < 0.16:
        return "black"
    if s < 0.18:
        return "white" if v > 0.85 else None
    if s < 0.35 or v < 0.3:
        return None
    return next(name for limit, name in _HUES if h < limit)


class Scene:
    def __init__(self, jpeg):
        image = Image.open(io.BytesIO(jpeg))
        image.draft("RGB", (GRID[0] * 4, GRID[1] * 4))

        self.pixels = list(image.convert("RGB").resize(GRID, Image.NEAREST).getdata())
        self.buckets = [(r >> 5, g >> 5, b >> 5) for r, g, b in self.pixels]
        self._counts = Counter(self.buckets)
        self.background = self._counts.most_common(1)[0][0]
        self._signature = self._salient = self._colours = self._ink = None


    def signature(self):
        if self._signature is None:
            counts = Counter((r >> 1, g >> 1, b >> 1) for r, g, b in self.buckets)
            self._signature = {k: n / len(self.buckets) for k, n in counts.items()}
        return self._signature

    def distance(self, other):
        a, b = self.signature(), other.signature()
        return sum(abs(a.get(k, 0) - b.get(k, 0)) for k in set(a) | set(b)) / 2


    def ink(self):
        """The cells that make up the screen's layout: text, buttons, small objects. Large even areas
        (the background, a page margin around the canvas) are left out: they stay when the menu goes."""
        if self._ink is None:
            limit = INK_SHARE * len(self.buckets)
            self._ink = [i for i, bucket in enumerate(self.buckets) if self._counts[bucket] < limit]
        return self._ink

    def ink_match(self, menu):
        """Share of `menu`'s non-background cells that this frame still shows; 0 when the menu has no layout."""
        ink = menu.ink()
        if len(ink) < MIN_INK * len(menu.buckets):
            return 0.0
        return sum(self.buckets[i] == menu.buckets[i] for i in ink) / len(ink)


    def _blobs(self, wanted):
        cols, rows = GRID
        blobs, seen = [], set()
        for start in range(len(wanted)):
            if not wanted[start] or start in seen:
                continue
            stack, cells = [start], []
            seen.add(start)
            while stack:
                cell = stack.pop()
                cells.append(cell)
                x, y = cell % cols, cell // cols
                for nx, ny in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                    neighbour = ny * cols + nx
                    if 0 <= nx < cols and 0 <= ny < rows and wanted[neighbour] and neighbour not in seen:
                        seen.add(neighbour)
                        stack.append(neighbour)
            blobs.append((len(cells), sum(c % cols + 0.5 for c in cells) / len(cells) / cols,
                          sum(c // cols + 0.5 for c in cells) / len(cells) / rows))
        return blobs

    def salient(self):
        """{'salient-near': (x, y), 'salient-big': (x, y)} as frame fractions; {} when nothing stands out."""
        if self._salient is None:
            limit = RARE_SHARE * len(self.buckets)
            blobs = self._blobs([self._counts[b] < limit for b in self.buckets])
            out = {}
            if blobs:
                out["salient-big"] = max(blobs)[1:]
                away = [(((x - 0.5) ** 2 + (y - 0.5) ** 2) ** 0.5, x, y) for _, x, y in blobs]
                away = [blob for blob in away if blob[0] >= CENTRE_HOLE]
                if away:
                    out["salient-near"] = min(away)[1:]
            self._salient = out
        return self._salient


    def colours(self):
        """{colour name: [(x, y), ...]} cell centres as frame fractions; the background colour is left out."""
        if self._colours is None:
            cols, rows = GRID
            out = {}
            for i, (pixel, bucket) in enumerate(zip(self.pixels, self.buckets)):
                if bucket == self.background:
                    continue
                name = colour_name(*pixel)
                if name:
                    out.setdefault(name, []).append(((i % cols + 0.5) / cols, (i // cols + 0.5) / rows))
            self._colours = out
        return self._colours

    def player(self, colour, previous):
        """The player's position: the cell of `colour` nearest where it was (the centre at first)."""
        anchor = previous or (0.5, 0.5)
        cells = self.colours().get(colour) if colour else None
        if not cells:
            return anchor
        near = min(cells, key=lambda c: (c[0] - anchor[0]) ** 2 + (c[1] - anchor[1]) ** 2)
        group = [c for c in cells if abs(c[0] - near[0]) < 0.04 and abs(c[1] - near[1]) < 0.06]
        return (sum(c[0] for c in group) / len(group), sum(c[1] for c in group) / len(group))

    def nearest(self, colour, origin, hole=0.0):
        """(distance, dx, dy) from `origin` to the nearest cell of `colour`, or None when it is not on screen.
        Cells closer than `hole` are left out: the player's own cells, when it has that colour."""
        cells = self.colours().get(colour)
        if cells and hole:
            cells = [c for c in cells if (c[0] - origin[0]) ** 2 + (c[1] - origin[1]) ** 2 >= hole * hole]
        if not cells:
            return None
        x, y = min(cells, key=lambda c: (c[0] - origin[0]) ** 2 + (c[1] - origin[1]) ** 2)
        dx, dy = x - origin[0], y - origin[1]
        return ((dx * dx + dy * dy) ** 0.5, dx, dy)


def direction_keys(dx, dy, keys4):
    """The direction keys for a move along (dx, dy); keys4 is (left, right, up, down)."""
    length = (dx * dx + dy * dy) ** 0.5
    if length < TOUCH:
        return []
    left, right, up, down = keys4
    out = []
    if abs(dx) >= AXIS_SHARE * length:
        out.append(right if dx > 0 else left)
    if abs(dy) >= AXIS_SHARE * length:
        out.append(down if dy > 0 else up)
    return out


In [ ]:
%%writefile /kaggle/working/o1/o1_actions.py
"""The generic action library for the two-tier agent, and the input state that plays it.

Nothing here knows a game. An option is one line of a small fixed grammar:

    wait                       let go of every key and button
    hold K | hold K1+K2        keep these keys down (the mouse stays as it is)
    tap K                      press K once (keys already held stay held)
    aim T                      move the pointer to T, no button (the keys stay as they are)
    fire T                     pointer at T with the left button held (the keys stay as they are)
    click T                    one left click at T
    toward C | away C          move toward / away from the nearest thing of colour C, with the direction
                               keys the controls name (code finds the colour and the player in the frame)

    T is a region (top-left, top, top-right, left, center, right, bottom-left, bottom, bottom-right),
    a position "X%,Y%" (across, down), or a salient target found in the frame by code:
    salient-near (the rare-colour blob nearest the screen centre) or salient-big (the largest one).
    C is a colour name from o1_perception.COLOURS.

Keys and mouse are two channels: a key option does not release the mouse and a mouse option does
not release the keys. That lets a picker that returns one option per step fly and fire at once.
The 255 options of the host's Drex Max starter are also built here, with the starter's own
labels and the starter's own actions, for the full-action-set arms.
"""
import re
from dataclasses import dataclass, field

from gga.protocol import Action, normalize_key, valid_key
from o1_perception import COLOURS, PLAYER_HOLE, direction_keys

MAX_OPTIONS = 16
REGIONS = {
    "top-left": (1 / 6, 1 / 6), "top": (1 / 2, 1 / 6), "top-right": (5 / 6, 1 / 6),
    "left": (1 / 6, 1 / 2), "center": (1 / 2, 1 / 2), "right": (5 / 6, 1 / 2),
    "bottom-left": (1 / 6, 5 / 6), "bottom": (1 / 2, 5 / 6), "bottom-right": (5 / 6, 5 / 6),
}
SALIENT = ("salient-near", "salient-big")
MOUSE_KINDS = ("aim", "fire", "click")

@dataclass(frozen=True)
class Option:
    label: str
    kind: str
    keys: tuple = ()
    target: object = None
    note: str = ""


@dataclass
class Plan:
    goal: str
    options: list
    source: str
    latency_s: float = 0.0
    raw: str = ""
    rejected: list = field(default_factory=list)
    screen: str = ""
    phase: str = ""
    player: str = ""
    targets: dict = field(default_factory=dict)


def parse_option(line):
    """One grammar line -> Option, or None when the line is not in the library."""
    text = line.strip().strip("-*•`'\" \t")
    text = re.sub(r"^\d+[.)]\s*", "", text)
    note = ""
    if "|" in text:
        text, note = (part.strip() for part in text.split("|", 1))
    words = text.split(None, 1)
    if not words:
        return None
    kind = words[0].lower().rstrip(":")
    kind = {"press": "tap", "move": "toward", "avoid": "away", "shoot": "fire"}.get(kind, kind)
    rest = words[1].strip() if len(words) > 1 else ""
    if kind == "wait" and not rest:
        return Option("wait", "wait", note=note[:80])
    if kind in ("hold", "tap"):
        keys = [normalize_key(k) for k in re.split(r"\s*\+\s*|\s+and\s+|\s*,\s*", rest) if k.strip()]
        if not keys or len(keys) > (1 if kind == "tap" else 3) or not all(valid_key(k) for k in keys):
            return None
        if len(set(keys)) != len(keys):
            return None
        return Option(f"{kind} {'+'.join(keys)}", kind, keys=tuple(keys), note=note[:80])
    if kind in ("toward", "away"):
        colour = rest.lower().replace("the ", "").strip()
        colour = {"gray": "white", "grey": "white", "violet": "purple", "magenta": "pink"}.get(colour, colour)
        return Option(f"{kind} {colour}", kind, target=colour, note=note[:80]) if colour in COLOURS else None
    if kind in MOUSE_KINDS:
        target = rest.lower().replace(" ", "")
        if target in ("centre", "middle"):
            target = "center"
        if target in REGIONS or target in SALIENT:
            return Option(f"{kind} {target}", kind, target=REGIONS.get(target, target), note=note[:80])
        match = re.fullmatch(r"(\d{1,3})%?,(\d{1,3})%?", target)
        if match:
            x, y = (min(max(int(v), 2), 98) for v in match.groups())
            return Option(f"{kind} {x}%,{y}%", kind, target=(x / 100, y / 100), note=note[:80])
    return None


def parse_options(lines):
    """Grammar lines -> at most MAX_OPTIONS distinct options, in the order given."""
    seen, out = set(), []
    for line in lines:
        option = parse_option(line)
        if option and option.label not in seen:
            seen.add(option.label)
            out.append(option)
        if len(out) == MAX_OPTIONS:
            break
    return out




_KEY_WORDS = {
    "space": ["Space"], "spacebar": ["Space"], "enter": ["Enter"], "return": ["Enter"],
    "shift": ["Shift"], "escape": ["Escape"], "tab": ["Tab"], "ctrl": ["Control"], "control": ["Control"],
}
_DIRECTION_SETS = {"arrow": ["ArrowLeft", "ArrowRight", "ArrowUp", "ArrowDown"], "wasd": ["a", "d", "w", "s"]}


def default_options(controls):
    """The fallback option list, from the controls text alone (no model, no screenshot)."""
    text = controls or ""
    low = text.lower()
    start, moves, mouse = [], [], []
    for word, keys in _KEY_WORDS.items():
        if re.search(rf"\b{word}\b", low):
            start += [f"tap {k}" for k in keys]
    if "any key" in low and "tap Space" not in start:
        start.append("tap Space")

    for letter in re.findall(r"(?<![A-Za-z'])([A-Z])(?![A-Za-z'])", text):
        if letter not in "AI" or re.search(rf"\b(press|key)\s+{letter}\b", text):
            start.append(f"tap {letter.lower()}")
    direction_sets = [keys for word, keys in _DIRECTION_SETS.items() if word in low]
    if "arrow" in low and "wasd" in low:
        direction_sets = direction_sets[:1]
    for keys in direction_sets:
        moves += [f"hold {k}" for k in keys]
    has_mouse = bool(re.search(r"\b(click|mouse|pointer|hover|drag|cursor)", low))
    if has_mouse:
        mouse += ["click salient-big", "click center"]
        if re.search(r"\b(aim|fire|shoot|hold the left)", low):
            mouse += ["fire salient-near", "fire left", "fire right", "fire top", "fire bottom"]
        else:
            mouse += ["click salient-near", "click left", "click right", "click top", "click bottom"]
    lines = start[:4] + moves + mouse + start[4:] + ["wait"]
    if len(parse_options(lines)) < 3:
        lines = ["tap Space", "tap Enter", "click center", "hold ArrowLeft", "hold ArrowRight",
                 "hold ArrowUp", "hold ArrowDown", "wait"]
    options = parse_options(lines)
    if options[-1].kind != "wait":
        options[-1] = Option("wait", "wait")
    return options


def movement_keys(controls):
    """(left, right, up, down) as the controls name them: WASD only when arrow keys are not offered."""
    low = (controls or "").lower()
    if "wasd" in low and "arrow" not in low:
        return ("a", "d", "w", "s")
    return ("ArrowLeft", "ArrowRight", "ArrowUp", "ArrowDown")





START_CLICK_GRID = [(x, y) for x in (50, 30, 70) for y in range(20, 93, 6)]


def start_candidates(controls):
    """Inputs that may start or restart a game, from the controls text alone, most likely first.

    Keys the text ties to starting come first. Clicks follow when the text says to click: a sweep
    down the middle of the screen, from the top (a start button's place is not in the text; the
    orchestrator's reading of the screen replaces this guess when it arrives).
    """
    text = controls or ""
    lines = []
    for sentence in re.split(r"(?<=[.;])\s+", text):
        low = sentence.lower()
        if not re.search(r"\b(start|begin|play|restart|again|continue)", low):
            continue
        for word, keys in _KEY_WORDS.items():
            if re.search(rf"\b{word}\b", low):
                lines += [f"tap {k}" for k in keys]
        if "any key" in low:
            lines.append("tap Space")
        for letter in re.findall(r"(?<![A-Za-z'])([A-Z])(?![A-Za-z'])", sentence):
            lines.append(f"tap {letter.lower()}")
        if "click" in low:
            lines += [f"click {x}%,{y}%" for x, y in START_CLICK_GRID]
    if not lines:
        lines = ["tap Space", "tap Enter"] + [f"click {x}%,{y}%" for x, y in START_CLICK_GRID[:13]]
    seen, out = set(), []
    for line in lines:
        option = parse_option(line)
        if option and option.label not in seen:
            seen.add(option.label)
            out.append(option)
    return out




STARTER_KEYS = [['Space'], ['Enter'], ['ArrowLeft'], ['ArrowRight'], ['ArrowUp'], ['ArrowDown'], ['w'], ['a'], ['s'],
                ['d'], ['Shift'], ['Control'], ['Escape'], ['Tab'], ['Backspace'],
                *[[k] for k in 'bcefghijklmnopqrtuvxyz0123456789'],
                *[[v, h] for v in ['ArrowUp', 'ArrowDown'] for h in ['ArrowLeft', 'ArrowRight']],
                *[[v, h] for v in 'ws' for h in 'ad'], *[[k, 'Space'] for k in ['ArrowLeft', 'ArrowRight', 'a', 'd']]]
STARTER_GRID = 15, 13


def starter_options():
    cells = [((c + 0.5) / STARTER_GRID[0], (r + 0.5) / STARTER_GRID[1])
             for r in range(STARTER_GRID[1]) for c in range(STARTER_GRID[0])]
    options = [Option("wait", "wait")]
    options += [Option("hold " + " and ".join(keys), "starter_hold", keys=tuple(keys)) for keys in STARTER_KEYS]
    options += [Option(f"click {x:.0%} {y:.0%}", "starter_click", target=(x, y)) for x, y in cells]
    assert len(options) == 255
    return options




class InputState:
    """What the agent holds: keys, mouse buttons and the pointer. One option changes one channel."""

    def __init__(self):
        self.keys, self.buttons, self.pointer = [], [], None

    def _step(self, width, height, duration_ms, keys=None, buttons=None):
        step = {"keys": list(self.keys if keys is None else keys),
                "buttons": list(self.buttons if buttons is None else buttons), "duration_ms": duration_ms}
        if self.pointer is not None:
            step["x"] = round(self.pointer[0] * (width - 1))
            step["y"] = round(self.pointer[1] * (height - 1))
        return step

    def play(self, option, width, height, scene, world):
        """The harness Action for this option.

        scene is the current frame's o1_perception.Scene. world is what the agent knows of the game:
        {"keys4": (left, right, up, down), "player": colour or None, "player_at": (x, y) or None}.
        """
        kind = option.kind
        if kind == "wait":
            self.keys, self.buttons = [], []
            return Action("wait", {"duration_ms": 1, "release_inputs": True})
        if kind == "starter_hold":
            self.keys, self.buttons = list(option.keys), []
            return Action("sequence", {"steps": [{"keys": [], "duration_ms": 50},
                                                 {"keys": list(option.keys), "duration_ms": 0}]})
        if kind == "starter_click":
            x, y = option.target
            return Action("click", {"x": round(x * (width - 1)), "y": round(y * (height - 1))})
        if kind == "hold":
            self.keys = list(option.keys)
            return Action("sequence", {"steps": [self._step(width, height, 0)]})
        if kind in ("toward", "away"):
            found = scene.nearest(option.target, world["player_at"] or (0.5, 0.5),
                                  PLAYER_HOLE if option.target == world["player"] else 0.0)
            if found is not None:
                sign = 1 if kind == "toward" else -1
                dx, dy = sign * found[1] * width, sign * found[2] * height
                scale = max(width, height)
                if kind == "away":
                    scale = ((dx * dx + dy * dy) ** 0.5) or 1.0
                    dx = dx or (scale if dy == 0 else 0.0)
                self.keys = direction_keys(dx / scale, dy / scale, world["keys4"])
            return Action("sequence", {"steps": [self._step(width, height, 0)]})
        if kind == "tap":
            key = option.keys[0]
            others = [k for k in self.keys if k != key]
            steps = [self._step(width, height, 100, keys=others + [key])]
            if key in self.keys:
                steps.insert(0, self._step(width, height, 50, keys=others))
            self.keys = others
            return Action("sequence", {"steps": steps + [self._step(width, height, 0)]})

        target = option.target
        if isinstance(target, str):
            target = scene.salient().get(target)
        if target is not None:
            self.pointer = (min(max(target[0], 0.0), 1.0), min(max(target[1], 0.0), 1.0))
        elif self.pointer is None:
            self.pointer = (0.5, 0.5)
        if kind == "aim":
            self.buttons = []
            return Action("sequence", {"steps": [self._step(width, height, 0)]})
        if kind == "fire":
            self.buttons = ["left"]
            return Action("sequence", {"steps": [self._step(width, height, 0)]})
        if kind == "click":
            self.buttons = []
            return Action("sequence", {"steps": [self._step(width, height, 50),
                                                 self._step(width, height, 100, buttons=["left"]),
                                                 self._step(width, height, 0)]})
        raise ValueError(f"unknown option kind {kind!r}")


In [ ]:
%%writefile /kaggle/working/o1/o1_pickers.py
"""Pickers: the fast tier. pick(options, context) returns the index of one option, every step.

context is a dict: jpeg (the current frame), scene (its o1_perception.Scene), description, controls,
goal (None when no orchestrator runs), notes (text the orchestrator read on the screens), previous (the
label played last step), static_steps (steps in a row the screen did not change), targets (colour ->
follow | take | avoid, from the orchestrator), player (its colour or None), player_at ((x, y) or None).

    RandomPicker   uniform over the list
    RulePicker     a code rule: play the orchestrator's first option, move down the list when the screen stalls
    ColourRulePicker  a code rule on the orchestrator's colour targets and the blob positions: no model per step
    DrexPicker     Drex Max: one forward pass gives every option a probability
"""
import io
import random
import time

from PIL import Image

from o1_perception import PLAYER_HOLE


DREX_IMAGE_SIDE = 448


STARTER_IMAGE_SIDE = 768
RULE_STALL_STEPS = 10
RULE_HOLD_STEPS = 5
AVOID_RADIUS = 0.12
TAKE_RADIUS = 0.15

STARTER_INSTRUCTIONS = (
    'You are playing the video game in the image. Choose the input that best makes progress in the game. '
    'hold X holds key X until your next choice, and choosing it again presses it again. '
    'click X% Y% clicks X% across and Y% down the screen. wait lets go of every key and button.')
LIST_INSTRUCTIONS = (
    'You are playing the video game in the image. Choose the input that best makes progress toward the goal '
    'right now. hold keeps keys down until another key option is chosen. tap presses a key once. '
    'aim moves the mouse pointer, fire holds the left mouse button with the pointer there, click clicks once. '
    'toward and away move with the direction keys relative to the nearest thing of that colour. '
    'A key option leaves the mouse as it is and a mouse option leaves the keys as they are. '
    'wait lets go of every key and button.')


class RandomPicker:
    def __init__(self, seed=0):
        self.seed = seed
        self.rng = random.Random(seed)

    def new_game(self):
        self.rng = random.Random(self.seed)

    def pick(self, options, context):
        return self.rng.randrange(len(options))


class RulePicker:
    """No model. The orchestrator orders its list (start or restart input first), so the first
    option is its best guess; when the screen has not changed for RULE_STALL_STEPS the rule tries the next."""

    def __init__(self):
        self.new_game()

    def new_game(self):
        self.index, self.held, self.labels = 0, 0, None

    def pick(self, options, context):
        labels = [option.label for option in options]
        if labels != self.labels:
            self.labels, self.index, self.held = labels, 0, 0
        self.held += 1
        if context["static_steps"] >= RULE_STALL_STEPS and self.held >= RULE_HOLD_STEPS:
            self.index, self.held = (self.index + 1) % len(options), 0
        return self.index


class ColourRulePicker:
    """The orchestrator reads what each colour means once; this rule turns that into a move every step.

    Order: move away from an "avoid" colour that is close; else go to a "take" colour that is close;
    else move toward the "follow" colour; else toward any "take" colour on screen. With no colour
    target known or on screen it falls back to RulePicker.
    """

    def __init__(self):
        self.fallback = RulePicker()

    def new_game(self):
        self.fallback.new_game()

    def pick(self, options, context):
        index = {option.label: i for i, option in enumerate(options)}
        scene, origin = context["scene"], context["player_at"] or (0.5, 0.5)
        found = {}
        for colour, role in context["targets"].items():
            label = f"{'away' if role == 'avoid' else 'toward'} {colour}"
            near = scene.nearest(colour, origin, PLAYER_HOLE if colour == context["player"] else 0.0)
            if near is not None and label in index:
                found.setdefault(role, []).append((near[0], index[label]))
        close_avoid = [hit for hit in found.get("avoid", []) if hit[0] < AVOID_RADIUS]
        if close_avoid:
            return min(close_avoid)[1]
        close_take = [hit for hit in found.get("take", []) if hit[0] < TAKE_RADIUS]
        if close_take:
            return min(close_take)[1]
        if found.get("follow"):
            return min(found["follow"])[1]
        if found.get("take"):
            return min(found["take"])[1]
        return self.fallback.pick(options, context)


class DrexPicker:
    """`model` is a drex_max.model.DecisionModel, or anything with its predict(rows) method.

    With goal None, STARTER_INSTRUCTIONS and STARTER_IMAGE_SIDE the request is the host starter's request,
    word for word. Every token costs time (about 3 ms per option), so the request carries the option
    labels only: no per-option notes and no screen text beyond the one-line goal.
    """

    def __init__(self, model, instructions=LIST_INSTRUCTIONS, image_side=DREX_IMAGE_SIDE):
        self.model, self.instructions, self.image_side = model, instructions, image_side
        self.probability, self.thought_s = 0.0, 0.0

    def new_game(self):
        pass

    def pick(self, options, context):
        start = time.time()
        frame = Image.open(io.BytesIO(context["jpeg"])).convert("RGB")
        frame.thumbnail((self.image_side, self.image_side))
        state = f"Brief: {context['description']}\nControls: {context['controls']}\n"
        if context["goal"]:
            state += f"Goal now: {context['goal']}\n"
        state += f"Previous action: {context['previous']}"
        question = {"type": "choice", "instructions": self.instructions,
                    "criteria": {option.label: None for option in options}}
        probabilities = self.model.predict([{"state": state, "question": question, "images": [frame]}])[0]
        if len(probabilities) != len(options):
            raise ValueError(f"{len(probabilities)} probabilities for {len(options)} options")
        best = max(range(len(options)), key=probabilities.__getitem__)
        self.probability, self.thought_s = probabilities[best], time.time() - start
        return best


In [ ]:
%%writefile /kaggle/working/o1/o1_agent.py
"""The two-tier agent (O1): a slow orchestrator reads the screen and writes the option list, a fast
picker picks every step, and a start step in code gets the game off its menu.

    TwoTierAgent(orchestrator, picker)                      the orchestrator's list, refreshed in a background thread
    TwoTierAgent(None, picker, options=default_options)     a fixed list built from the controls text, no orchestrator
    TwoTierAgent(None, picker, options=starter_options(), start=False)
                                                            the host starter's 255 options, nothing else

step() never waits for the orchestrator. Until its first answer the list is default_options(controls);
when it fails or is slow, the last list stays. There is no sleep: a reply goes out as soon as it is ready.

The start step. The first frame of a game is taken as its opening screen (title, rules, PLAY). While the
current frame still shows that screen's layout (Scene.ink_match), the agent plays start candidates, one
every START_EVERY steps: the input that left this screen last time, the orchestrator's first option for a
menu or game-over screen, then start_candidates(controls). The picker takes over on the first frame that
no longer matches. A game that dies back to the same screen is caught on the first frame, with no model call.
A screen the orchestrator labels menu or gameover is remembered as a menu screen too, on trial: if the
start candidates do not get the game off it within START_PATIENCE_S, it was not a menu (or not one we
can leave) and it is forgotten. The opening screen is never forgotten.
"""
import json
import os
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor

from gga.protocol import Action
from o1_actions import (MAX_OPTIONS, InputState, Plan, default_options, movement_keys, parse_option,
                        start_candidates)
from o1_perception import MIN_INK, Scene


REFRESH_S = 25.0
MIN_GAP_S = 4.0
CHANGE_DISTANCE = 0.30
STATIC_DISTANCE = 0.004
STALL_STEPS = 15
MENU_MATCH = 0.6
OPENING_WINDOW_S = 3.0
START_EVERY = 2
START_PATIENCE_S = 10.0
MENU_STATIC_STEPS = 3
MAX_LEARNED_MENUS = 2
NOTES_CHARS = 600


class TwoTierAgent:
    def __init__(self, orchestrator, picker, options=None, start=True):
        self.orchestrator, self.picker, self.start = orchestrator, picker, start
        self.fixed_options = options
        self.pool = ThreadPoolExecutor(1, thread_name_prefix="orchestrator") if orchestrator else None
        self.trace_path = os.environ.get("O1_TRACE")
        self.last_elapsed = float("inf")
        self.step_s = 0.0
        self.game = 0
        self._new_game(None)


    def _new_game(self, info):
        self.game += 1
        self.inputs = InputState()
        self.previous = "none"
        self.steps = 0
        self.static_steps = 0
        self.last_scene = None
        self.pending = None
        self.asked_at = None
        self.retry = False
        self.asked_on_menu = None
        self.plan_scene = None
        self.plan = None
        self.menus = []
        self.learned = 0
        self.menu_now = None
        self.opening = None
        self.on_menu = False
        self.menu_since = 0.0
        self.start_index = 0
        self.start_option = None
        self.worked = None
        self.last_candidate = None
        self.notes = ""
        self.targets = {}
        self.world = {"keys4": movement_keys(info["controls"] if info else ""), "player": None, "player_at": None}
        self.candidates = start_candidates(info["controls"]) if info else []
        if info is not None:
            if callable(self.fixed_options):
                options = self.fixed_options(info["controls"])
            else:
                options = self.fixed_options or default_options(info["controls"])
            self.plan = Plan(None, options, "fixed" if self.fixed_options else "default")
            self._trace({"event": "plan", "t": 0.0, "source": self.plan.source,
                         "options": [o.label for o in options]})
        self.picker.new_game()

    def _trace(self, record):
        if self.trace_path:
            with open(self.trace_path, "a") as file:
                file.write(json.dumps({"game": self.game, **record}) + "\n")


    def _take(self, plan, asked_scene, asked_static, t, asked_at):
        """A new plan: the list, the goal, and what the orchestrator read off the screen."""
        if plan.screen and plan.screen not in self.notes:
            self.notes = (self.notes + " / " + plan.screen).strip(" /")[:NOTES_CHARS]
        if plan.player:
            self.world["player"] = plan.player
        self.targets.update(plan.targets)
        if plan.phase in ("menu", "gameover"):
            self.start_option = plan.options[0]
            known = any(asked_scene.ink_match(menu) >= MENU_MATCH for menu in self.menus)


            if not known and asked_static and self.learned < MAX_LEARNED_MENUS \
                    and len(asked_scene.ink()) >= MIN_INK * len(asked_scene.buckets):
                self.menus.append(asked_scene)
                self.learned += 1
                self._trace({"event": "menu_learned", "t": t})
        labels = {option.label for option in plan.options}
        extra = []
        for colour, role in self.targets.items():
            option = parse_option(f"{'away' if role == 'avoid' else 'toward'} {colour} | {role}")
            if option.label not in labels:
                extra.append(option)
        plan.options = (plan.options[:1] + extra + plan.options[1:])[:MAX_OPTIONS]
        self.plan, self.plan_scene = plan, asked_scene
        self._trace({"event": "plan", "t": t, "asked_t": round(asked_at, 2), "source": plan.source,
                     "latency_s": round(plan.latency_s, 2), "phase": plan.phase, "goal": plan.goal,
                     "screen": plan.screen, "player": plan.player, "targets": plan.targets,
                     "options": [o.label for o in plan.options], "rejected": plan.rejected})

    def _orchestrate(self, obs, scene):
        t = round(obs.elapsed_s, 2)
        if self.pending is not None:
            future, game, asked_at, asked_scene, asked_static = self.pending
            if not future.done():
                return
            self.pending = None
            if game == self.game:
                try:
                    self._take(future.result(), asked_scene, asked_static, t, asked_at)
                except Exception as exc:
                    self.plan_scene = asked_scene
                    self.asked_at, self.retry = obs.elapsed_s, True
                    self._trace({"event": "plan_failed", "t": t, "asked_t": round(asked_at, 2),
                                 "error": f"{type(exc).__name__}: {exc}"[:400]})
        since = float("inf") if self.asked_at is None else obs.elapsed_s - self.asked_at
        why = None
        if since >= REFRESH_S:
            why = "timer"
        elif since >= MIN_GAP_S:
            if self.retry:
                why = "retry after a failure"
            elif self.asked_on_menu is not None and self.on_menu != self.asked_on_menu:
                why = "menu appeared" if self.on_menu else "menu left"
            elif self.plan_scene is not None and scene.distance(self.plan_scene) > CHANGE_DISTANCE:
                why = "screen changed"
            elif self.static_steps >= STALL_STEPS:
                why = "screen stalled"
        if why:
            self.retry = False
            self.asked_at, self.asked_on_menu = obs.elapsed_s, self.on_menu
            future = self.pool.submit(self.orchestrator.propose, obs.info["description"], obs.info["controls"],
                                      obs.image, self.plan.goal, self.notes)
            self.pending = (future, self.game, obs.elapsed_s, scene, self.static_steps >= MENU_STATIC_STEPS)
            self._trace({"event": "ask", "t": t, "why": why})


    def _menu_state(self, obs, scene):
        if not self.menus and obs.elapsed_s <= OPENING_WINDOW_S and len(scene.ink()) >= MIN_INK * len(scene.buckets):
            self.menus.append(scene)
            self.opening = scene
        self.menu_now = next((menu for menu in self.menus if scene.ink_match(menu) >= MENU_MATCH), None)
        if self.menu_now is not None and self.menu_now is not self.opening and self.on_menu \
                and obs.elapsed_s - self.menu_since >= START_PATIENCE_S:
            self.menus.remove(self.menu_now)
            self.menu_now = None
            self._trace({"event": "menu_forgotten", "t": round(obs.elapsed_s, 2)})
        on_menu = self.menu_now is not None
        if on_menu and not self.on_menu:
            self.menu_since, self.start_index = obs.elapsed_s, 0
            self._trace({"event": "menu", "t": round(obs.elapsed_s, 2)})
        if self.on_menu and not on_menu:
            self.worked = self.last_candidate
            self._trace({"event": "menu_left", "t": round(obs.elapsed_s, 2), "after_s": round(
                obs.elapsed_s - self.menu_since, 2), "last_candidate": self.worked.label if self.worked else None})
        self.on_menu = on_menu

    def _start_candidate(self, obs):
        """The start candidate due at this step, or None when it is not a candidate's turn."""
        waited = obs.elapsed_s - self.menu_since
        if self.steps % (START_EVERY if waited < START_PATIENCE_S else 5):
            return None
        seen, ordered = set(), []
        for option in [self.worked, self.start_option, *self.candidates]:
            if option is not None and option.label not in seen:
                seen.add(option.label)
                ordered.append(option)
        option = ordered[self.start_index % len(ordered)]
        self.start_index += 1
        self.last_candidate = option
        return option


    def step(self, obs, tools):
        began = time.time()
        if obs.elapsed_s < self.last_elapsed:
            self._new_game(obs.info)
        self.last_elapsed = obs.elapsed_s
        self.steps += 1
        scene = Scene(obs.image)
        if self.last_scene is not None and scene.distance(self.last_scene) < STATIC_DISTANCE:
            self.static_steps += 1
        else:
            self.static_steps = 0
        self.last_scene = scene
        self._menu_state(obs, scene)
        if self.orchestrator is not None:
            self._orchestrate(obs, scene)
        if self.world["player"]:
            self.world["player_at"] = scene.player(self.world["player"], self.world["player_at"])
        options = self.plan.options
        mode, option = "pick", None
        if self.start and self.on_menu:
            option = self._start_candidate(obs)
            mode = "start" if option else "pick"
            if option is None and obs.elapsed_s - self.menu_since < START_PATIENCE_S:
                mode = "hold"
        if mode == "pick":
            try:
                index = self.picker.pick(options, {
                    "jpeg": obs.image, "scene": scene, "description": obs.info["description"],
                    "controls": obs.info["controls"], "goal": self.plan.goal, "notes": self.notes,
                    "previous": self.previous, "static_steps": self.static_steps, "targets": self.targets,
                    "player": self.world["player"], "player_at": self.world["player_at"]})
            except Exception:
                print(traceback.format_exc(), file=sys.stderr, flush=True)
                self._trace({"event": "pick_failed", "t": round(obs.elapsed_s, 2),
                             "error": traceback.format_exc()[-400:]})
                index = 0
            option = options[index]
        if option is None:
            action, label = Action("skip"), "skip"
        else:
            action = self.inputs.play(option, obs.info["width"], obs.info["height"], scene, self.world)
            label = self.previous = option.label
        last = obs.last_action or {}
        self._trace({"event": "step", "t": round(obs.elapsed_s, 2), "mode": mode, "pick": label, "n": len(options),
                     "menu": self.on_menu, "ms": round(1000 * (time.time() - began), 1), "last_ok": last.get("ok"),
                     "last_error": last.get("error"), "lat": last.get("wall_latency_s")})
        self.step_s = time.time() - began
        return action


In [ ]:
MODEL_AGENT = QwenAgent

import time
from o1_actions import starter_options
from o1_agent import OPENING_WINDOW_S, START_PATIENCE_S, STATIC_DISTANCE, TwoTierAgent
from o1_perception import Scene


class _NoPicker:
    def new_game(self):
        pass


START_HANDOVER_S = 6.0
REENTER_STATIC = 2


class Agent(TwoTierAgent):
    """The start step from o1_agent in front of the Qwen agent (MODEL_AGENT).

    Changed: the opening screen used to stay a "menu" for the whole game, so in a game without a title screen whose
    layout never changes (board, puzzle, clicker UI) the start step played alone for 10 s and then injected a start
    probe (Space, Enter, a click down the middle) every 5th step for the rest of the game. Now each menu episode gets
    START_HANDOVER_S; after that Qwen plays alone, and the start step comes back only after the game really left the
    opening screen and returned to it (game over -> title) and that screen stands still."""
    def __init__(self):
        super().__init__(None, _NoPicker(), options=starter_options())
        self.gemma = MODEL_AGENT()

    def _new_game(self, info):
        super()._new_game(info)
        self.start_open = False
        self.episode_used = False
        self.left_menu = False
        if getattr(self, 'gemma', None) is not None:
            self.gemma.last_action = 'none'

    def step(self, obs, tools):
        began = time.time()
        if obs.elapsed_s < self.last_elapsed:
            self._new_game(obs.info)
        self.last_elapsed = obs.elapsed_s
        self.steps += 1
        scene = Scene(obs.image)
        self.static_steps = self.static_steps + 1 if (self.last_scene is not None and
                                                      scene.distance(self.last_scene) < STATIC_DISTANCE) else 0
        self.last_scene = scene
        was = self.on_menu
        self._menu_state(obs, scene)
        if was and not self.on_menu:
            self.left_menu = True
        if self.on_menu and not was:
            self.episode_used = False
            self.start_open = obs.elapsed_s <= OPENING_WINDOW_S + 0.5
        if (self.on_menu and not self.start_open and not self.episode_used and self.left_menu
                and self.static_steps >= REENTER_STATIC):
            self.start_open, self.menu_since, self.start_index = True, obs.elapsed_s, 0
        if self.start_open and (not self.on_menu or obs.elapsed_s - self.menu_since >= START_HANDOVER_S):
            self.start_open, self.episode_used = False, True
        if self.start_open:
            option = self._start_candidate(obs)
            if option is not None:
                self.previous = option.label
                self.step_s = time.time() - began
                return self.inputs.play(option, obs.info['width'], obs.info['height'], scene, self.world)
            self.step_s = time.time() - began
            return Action('skip')
        action = self.gemma.step(obs, tools)
        self.previous = self.gemma.last_action
        self.step_s = time.time() - began
        return action


    last_action = property(lambda self: self.previous)
    thought_s = property(lambda self: self.step_s)


## Scripted solvers (sample games)

Each solver reads only the screenshot (numpy and PIL) and answers in milliseconds; each scores at or near 1.0 across
seeds in local 10-minute runs.

| File | Game |
|---|---|
| `dispatch.py` | Picks the solver for a game, or the fallback agent for any other game |
| `asteroid_capitalist.py` | Shooter: dodges by simulating the ship's physics, leads its shots, recognises and fits the boss |
| `blarglike.py` | Grid dungeon: reads the tiles and battery, plans flower catches over their random steps |
| `cavebots.py` | Seals the Core in a pocket whose attack stop cells stay on the grass, rebuilds the plug, flees a breach |
| `off_line.py` | Plans arrow-key moves around the falling squares, its speed rule checked against measured falls |
| `generic.py` | Fallback when no other agent is given (local runs) |
| `scripted_agent.py` | The scripted solvers alone, as an agent file for the smoke play |

In [ ]:
os.makedirs('/kaggle/working/solvers', exist_ok=True)
sys.path.insert(0, '/kaggle/working')


In [ ]:
%%writefile /kaggle/working/solvers/dispatch.py
"""Plays the sample games with their scripted solvers and every other game with `fallback`."""
from __future__ import annotations

import importlib

from gga.protocol import Action, Observation, ToolSpec

SOLVERS = {
    "asteroid_capitalist": "solvers.asteroid_capitalist",
    "blarglike": "solvers.blarglike",
    "cavebots": "solvers.cavebots",
    "off_line": "solvers.off_line",
}

BRIEFS = {
    "asteroid_capitalist": "Fly home through a lawless solar system",
    "blarglike": "Traverse a turn-based dungeon, gathering nutrients",
    "cavebots": "Dig out from your Core's tunnel",
    "off_line": "Stay on the green line to preserve health",
}
RESTARTS = ("reload", "relaunch")


def game_of(obs: Observation) -> str | None:
    """The sample game this observation is from, or None for any other game."""
    game_id = obs.info.get("game_id")
    if game_id in SOLVERS:
        return game_id
    description = obs.info.get("description") or ""
    return next((g for g, brief in BRIEFS.items() if description.startswith(brief)), None)


class Agent:
    def __init__(self, fallback=None) -> None:


        self.fallback = fallback
        self.last_elapsed = float("inf")
        self.solver = None

    def _new_solver(self, obs: Observation):
        name = SOLVERS.get(game_of(obs))
        if name is None:
            return self.fallback or importlib.import_module("solvers.generic").Solver()
        try:
            return importlib.import_module(name).Solver()
        except Exception as e:
            print(f"solver {name} failed to load: {e!r}", flush=True)
            return self.fallback or importlib.import_module("solvers.generic").Solver()

    def step(self, obs: Observation, tools: list[ToolSpec]) -> Action:
        restarted = any(d.get("code") in RESTARTS for d in (obs.last_action or {}).get("dropped", []))
        new_game = (self.solver is None or obs.elapsed_s < self.last_elapsed
                    or (obs.elapsed_s == self.last_elapsed and not obs.last_action))
        if new_game or (restarted and self.solver is not self.fallback):
            self.solver = self._new_solver(obs)
        self.last_elapsed = obs.elapsed_s
        try:
            return self.solver.step(obs, tools)
        except Exception as e:
            print(f"solver step failed: {e!r}", flush=True)
            return Action("skip", {})


In [ ]:
%%writefile /kaggle/working/solvers/__init__.py
"""Scripted solvers for the GGA sample games, and the dispatcher that picks one."""


In [ ]:
%%writefile /kaggle/working/solvers/asteroid_capitalist.py
"""Asteroid Capitalist: hold fire, aim at the nearest enemy with lead, steer by a short-horizon threat model.

Screen = game / 1.2 (the 1920x1080 game is drawn into the top 1600x900 of the viewport).
Red ships are enemies, blue lasers are enemy bullets, the large green blob below the HUD is us.
"""
from __future__ import annotations

import io
import math
import os

import numpy as np
from PIL import Image

from gga.protocol import Action

SCALE = 1 / 1.2
W, H = 1600, 900
LO, HI_X, HI_Y = 50 * SCALE, W - 50 * SCALE, H - 50 * SCALE
ACC, VMAX, DRAG = 1.5, 7.5, 0.005
BULLET_V = 15 * 60 * SCALE
ENEMY_V = -3 * 60 * SCALE
R_SHIP, R_ENEMY, R_BULLET, R_BOSS = 35 * SCALE, 35 * SCALE, 5 * SCALE, 100 * SCALE
PLAY = (800, 625)
RESPAWN = (100 * SCALE, 300 * SCALE)
HOME_X = 190


BOSS_W, BOSS_CELLS, BOSS_SEEN, BOSS_R = 140, 60, 3, 170
POWERUP_TURNS = 5
STANDOFF = float(os.environ.get("AC_STANDOFF", 0.05))
STANDOFF_PX = float(os.environ.get("AC_STANDOFF_PX", 200))
HOME_W = float(os.environ.get("AC_HOME_W", 0.001))
BOSS_GAP = 650
DT = 1 / 60
TURN = 0.2
DEBUG = os.environ.get("AC_DEBUG")
CELL = 4

DIRS = {(0, 0): [], (0, -1): ["w"], (0, 1): ["s"], (-1, 0): ["a"], (1, 0): ["d"],
        (-1, -1): ["w", "a"], (1, -1): ["w", "d"], (-1, 1): ["s", "a"], (1, 1): ["s", "d"]}


def _decode(jpeg):
    img = Image.open(io.BytesIO(jpeg))
    img.draft("RGB", (800, 500))
    a = np.asarray(img.convert("RGB"), dtype=np.int16)
    if a.shape[1] != 800:
        a = np.asarray(img.convert("RGB").resize((800, 500)), dtype=np.int16)
    return a[:450]


def _components(mask, min_cells=1, merge=1):
    """Connected blobs on a CELL grid: list of (x, y, cells, w, h) in screen px."""
    h, w = mask.shape
    g = mask[: h - h % CELL, : w - w % CELL].reshape(h // CELL, CELL, w // CELL, CELL).sum((1, 3)) >= 2
    ys, xs = np.nonzero(g)
    occupied = set(zip(ys.tolist(), xs.tolist()))
    blobs = []
    while occupied:
        start = occupied.pop()
        stack, cells = [start], [start]
        while stack:
            cy, cx = stack.pop()
            for dy in range(-merge, merge + 1):
                for dx in range(-merge, merge + 1):
                    n = (cy + dy, cx + dx)
                    if n in occupied:
                        occupied.remove(n)
                        stack.append(n)
                        cells.append(n)
        if len(cells) >= min_cells:
            cy = np.array([c[0] for c in cells])
            cx = np.array([c[1] for c in cells])
            k = 2 * CELL
            blobs.append(((cx.mean() + 0.5) * k, (cy.mean() + 0.5) * k, len(cells),
                          (cx.max() - cx.min() + 1) * k, (cy.max() - cy.min() + 1) * k))
    return blobs


def _merge(blobs, within):
    """Join blobs closer than `within` (a ship's red wings come out as two blobs)."""
    blobs = [list(b) for b in blobs]
    merged = True
    while merged:
        merged = False
        for i in range(len(blobs)):
            for j in range(i + 1, len(blobs)):
                a, b = blobs[i], blobs[j]
                if math.hypot(a[0] - b[0], a[1] - b[1]) < within:
                    n = a[2] + b[2]
                    x0 = min(a[0] - a[3] / 2, b[0] - b[3] / 2)
                    x1 = max(a[0] + a[3] / 2, b[0] + b[3] / 2)
                    y0 = min(a[1] - a[4] / 2, b[1] - b[4] / 2)
                    y1 = max(a[1] + a[4] / 2, b[1] + b[4] / 2)
                    blobs[i] = [(a[0] * a[2] + b[0] * b[2]) / n, (a[1] * a[2] + b[1] * b[2]) / n, n,
                                x1 - x0, y1 - y0]
                    del blobs[j]
                    merged = True
                    break
            if merged:
                break
    return [tuple(b) for b in blobs]


def _masks(a):
    R, G, B = a[..., 0], a[..., 1], a[..., 2]
    red = (R > 150) & (G < 100) & (B < 100) & (R - G > 100)
    blue = (B > 200) & (G > 140) & (R < 120)
    green = (G > 160) & (R < 150) & (B < 110) & (G - R > 50)
    return red, blue, green


def _boss_and_powerups(a, red):
    R, G, B = a[..., 0], a[..., 1], a[..., 2]
    dark = (R > 70) & (R < 160) & (G < 50) & (B < 55) & (R - G > 45)
    reds = [b for b in _components(dark | red, 1, merge=1) if b[2] >= 3]
    yellow = (R > 220) & (G > 170) & (B < 70)
    return reds, [b for b in _components(yellow, 1, merge=1) if b[2] >= 5]


def _perceive(jpeg):
    a = _decode(jpeg)
    red, blue, green = _masks(a)
    life = bool(green[5:20, 3:23].sum() > 10)
    green[:23, :280] = False
    enemies = _merge(_components(red, 2), 45)
    bullets = _components(blue, 1, merge=1)
    greens = _components(green, 1, merge=2)
    ships = [b for b in greens if b[2] >= 6]
    reds, powerups = _boss_and_powerups(a, red)
    return enemies, bullets, ships, life, reds, powerups


def _match(now, before, dt, lo, hi, default):
    """Velocity per blob in `now` from its nearest neighbour in `before`."""
    out = []
    for x, y, n, w, h in now:
        best, v = None, default
        for px, py, *_ in before:
            d = math.hypot(x - px, y - py)
            if lo <= d <= hi and (best is None or d < best):
                best, v = d, ((x - px) / dt, (y - py) / dt)
        out.append((x, y, v[0] if v else None, v[1] if v else None, max(w, h)))
    return out


def _sim(pos, vel, plan):
    """Ship positions every 1/60 s under `plan` [(direction, seconds), ...]; game physics in screen px."""
    x, y = pos[0] / SCALE, pos[1] / SCALE
    vx, vy = vel[0] / SCALE / 60, vel[1] / SCALE / 60
    out = []
    for (dx, dy), secs in plan:
        for _ in range(int(round(secs * 60))):
            ax, ay = ACC * dx, ACC * dy
            sp = math.hypot(vx, vy)
            if sp > 0:
                ax -= DRAG * sp * vx
                ay -= DRAG * sp * vy
            vx, vy = vx + ax, vy + ay
            sp = math.hypot(vx, vy)
            if sp > VMAX:
                vx, vy = vx * VMAX / sp, vy * VMAX / sp
            if vx * vx + vy * vy < 0.5:
                vx = vy = 0.0
            x = min(max(x + vx, 50), 1920 - 50)
            y = min(max(y + vy, 50), 1080 - 50)
            out.append((x * SCALE, y * SCALE, vx * SCALE * 60, vy * SCALE * 60))
    return out


class Solver:
    def __init__(self):
        self.pos = (W / 2, H / 2)
        self.vel = (0.0, 0.0)
        self.cmd = (0, 0)
        self.no_life = 0
        self.turn = 0
        self.lost = 0
        self.home_x, self.powerups = HOME_X, []
        self.boss_hist = []
        self.boss_seen = 0
        self.boss_last = None
        self.boss_missing = 0
        self.yellow_hist = []

    def step(self, obs, tools):
        self.turn += 1
        enemies, bullets, ships, life, reds, powerups = _perceive(obs.image)
        self.now = obs.elapsed_s
        if not life:
            self.no_life += 1
            if self.no_life >= 2 and self.turn % 3 == 0:
                self.pos, self.vel, self.cmd = (W / 2, H / 2), (0.0, 0.0), (0, 0)
                self.boss_seen, self.boss_last, self.boss_hist = 0, None, []
                return Action("click", {"x": PLAY[0], "y": PLAY[1]})
            return Action("skip", {})
        self.no_life = 0


        near = [b for b in ships if math.hypot(b[0] - self.pos[0], b[1] - self.pos[1]) < 120]
        if self.lost <= 6 and near:
            ship = max(near, key=lambda b: b[2])
        else:
            ship = max((b for b in ships if b[2] >= 12), key=lambda b: b[2], default=None)
        if ship is not None:
            sx, sy = ship[0], ship[1]
            px, py = self.pos
            if math.hypot(sx - px, sy - py) > 150:
                self.vel = (0.0, 0.0)
            self.pos = (sx, sy)
            self.lost = 0
        else:
            self.lost += 1
            if self.lost > 6:
                self.pos, self.vel = RESPAWN, (0.0, 0.0)


        frames = obs.frames or []
        if len(frames) >= 2:
            a = _decode(frames[-2][1])
            red_b, blue_b, _ = _masks(a)
            bullets_v = _match(bullets, _components(blue_b, 1, merge=1), 0.05, 10, 70, None)
        else:
            bullets_v = [(x, y, None, None, max(w, h)) for x, y, n, w, h in bullets]
        bullets_v = [b if b[2] is not None else self._aimed(b) for b in bullets_v]
        if len(frames) >= 4:
            a = _decode(frames[0][1])
            red_b, _, _ = _masks(a)
            enemies_v = _match(enemies, _merge(_components(red_b, 2), 45), 0.15, 3, 60, (ENEMY_V, 0.0))
        else:
            enemies_v = [(x, y, ENEMY_V, 0.0, max(w, h)) for x, y, n, w, h in enemies]

        boss = self._find_boss(reds)
        if boss is not None:
            enemies_v = [e for e in enemies_v if math.hypot(e[0] - boss[0], e[1] - boss[1]) > BOSS_R]
            self.boss_hist = [h for h in self.boss_hist if obs.elapsed_s - h[0] < 12] + [(obs.elapsed_s, boss[0], boss[1])]
            bx, by, bvx, bvy = self._boss_predict(obs.elapsed_s)
            enemies_v.append((bx, by, bvx, bvy, 200))
        elif self.boss_last is None:
            self.boss_hist = []
        self.home_x = HOME_X if boss is None else min(max(HOME_X, boss[0] - BOSS_GAP), W / 2)
        self.powerups = self._still(powerups)
        cmd = self._choose(enemies_v, bullets_v)

        traj = _sim(self.pos, self.vel, [(self.cmd, TURN)])
        self.cmd = cmd
        self.pos, self.vel = (traj[-1][0], traj[-1][1]), (traj[-1][2], traj[-1][3])

        aim = self._aim(enemies_v, (traj[-1][0], traj[-1][1]))
        keys = DIRS[cmd]
        if DEBUG:
            if self.turn % (1 if os.environ.get("AC_DEBUG_ALL") else 5) == 0:
                open(f"{DEBUG}/{self.turn:05d}.jpg", "wb").write(obs.image)
            print(f"t={obs.elapsed_s:.1f} ship={ship and (int(ship[0]), int(ship[1]))} e={len(enemies)} "
                  f"b={len(bullets)} cmd={cmd} aim={int(aim[0]), int(aim[1])}", flush=True)
        return Action("sequence", {"steps": [{"keys": keys, "buttons": ["left"],
                                              "x": int(aim[0]), "y": int(aim[1]), "duration_ms": 0}]})

    def _find_boss(self, reds):
        """The boss's centre, from its look: a red part wider than any ship, on BOSS_SEEN turns
        running; once confirmed, all red near where it was (its two wings, or one tinted body)."""
        wide = [b for b in reds if b[3] >= BOSS_W and b[2] >= BOSS_CELLS]
        self.boss_seen = self.boss_seen + 1 if wide else 0
        if self.boss_last is None:
            if self.boss_seen < BOSS_SEEN:
                return None
            self.boss_last = max(wide, key=lambda b: b[2])[:2]
        near = [b for b in reds if math.hypot(b[0] - self.boss_last[0], b[1] - self.boss_last[1]) < BOSS_R]
        if not near:
            self.boss_missing += 1
            if self.boss_missing > 10:
                self.boss_last, self.boss_hist = None, []
            return None
        self.boss_missing = 0
        n = sum(b[2] for b in near)
        boss = (sum(b[0] * b[2] for b in near) / n, sum(b[1] * b[2] for b in near) / n, n, 200, 200)
        self.boss_last = boss[:2]
        return boss

    def _still(self, yellows):
        """Yellow blobs that have sat in the same place for POWERUP_TURNS turns: pickups."""
        self.yellow_hist = (self.yellow_hist + [yellows])[-POWERUP_TURNS:]
        if len(self.yellow_hist) < POWERUP_TURNS:
            return []
        return [p for p in yellows
                if all(any(math.hypot(p[0] - q[0], p[1] - q[1]) < 12 for q in past) for past in self.yellow_hist)]

    def _boss_predict(self, now):
        """Boss position and velocity now: x drifts as measured, y bobs on c + a sin(wt) + b cos(wt)."""
        hist = self.boss_hist
        x = hist[-1][1]
        recent = [h for h in hist if now - h[0] < 1.5]
        vx = 0.0
        if len(recent) >= 4 and recent[-1][0] > recent[0][0]:
            vx = (recent[-1][1] - recent[0][1]) / (recent[-1][0] - recent[0][0])
            vx = min(max(vx, 2 * ENEMY_V), 0.0) if abs(vx) > 30 else 0.0
        if len(hist) < 8:
            return x, hist[-1][2], vx, 0.0
        t = np.array([h[0] for h in hist])
        y = np.array([h[2] for h in hist])
        best = None
        for w in (0.54, 0.57, 0.6, 0.63, 0.66):
            A = np.stack([np.ones_like(t), np.sin(w * t), np.cos(w * t)], 1)
            coef, *_ = np.linalg.lstsq(A, y, rcond=None)
            res = float(((A @ coef - y) ** 2).sum())
            if best is None or res < best[0]:
                best = (res, w, coef)
        _, w, (c, a, b) = best
        py = c + a * math.sin(w * now) + b * math.cos(w * now)
        vy = w * (a * math.cos(w * now) - b * math.sin(w * now))
        return x, py, vx, vy

    def _aimed(self, b):
        """An unmatched enemy laser: assume it was fired at where we are."""
        x, y, _, _, size = b
        dx, dy = self.pos[0] - x, self.pos[1] - y
        n = math.hypot(dx, dy) or 1.0
        return (x, y, dx / n * BULLET_V, dy / n * BULLET_V, size)

    def _aim(self, enemies, me):
        best, target = None, None
        for x, y, vx, vy, size in enemies:

            x, y = x + vx * TURN, y + vy * TURN
            if x < me[0] - 40 or x > W + 20:
                continue
            d = math.hypot(x - me[0], y - me[1])
            if best is None or d < best:
                best, target = d, (x, y, vx, vy)
        if target is None:
            return (min(me[0] + 600, W - 50), me[1])
        x, y, vx, vy = target
        t = 0.0
        if self.boss_hist and len(self.boss_hist) >= 8 and best > 300 and abs(x - self.boss_hist[-1][1]) < 120:
            for _ in range(4):
                tx, ty, _, _ = self._boss_predict(self.now + TURN + t)
                t = math.hypot(tx - me[0], ty - me[1]) / BULLET_V
            tx, ty, _, _ = self._boss_predict(self.now + TURN + t)
        else:
            for _ in range(3):
                t = math.hypot(x + vx * t - me[0], y + vy * t - me[1]) / BULLET_V
            tx, ty = x + vx * t, y + vy * t

        dx, dy = tx - me[0], ty - me[1]
        n = math.hypot(dx, dy) or 1.0
        for r in (n, 300, 200, 120):
            ax, ay = me[0] + dx / n * r, me[1] + dy / n * r
            if LO <= ax <= HI_X and LO <= ay <= HI_Y:
                return (ax, ay)
        return (min(max(tx, LO), HI_X), min(max(ty, LO), HI_Y))

    def _choose(self, enemies, bullets):
        threats = []
        for x, y, vx, vy, size in bullets:
            threats.append((x, y, vx, vy, R_BULLET + R_SHIP, 6.0, 25.0, 0.0))
        for x, y, vx, vy, size in enemies:
            r = R_BOSS if size > 120 else R_ENEMY
            threats.append((x, y, vx, vy, r + R_SHIP, 10.0, 60.0, 0.0 if size > 120 else 1.0))
        best, choice = None, self.cmd
        for d in DIRS:
            costs = []
            for tail in ((d, 0.6), ((0, 0), 0.6)):
                traj = _sim(self.pos, self.vel, [(self.cmd, TURN), (d, TURN), tail])
                costs.append(self._cost(traj, threats))
            cost = min(costs)
            x, y = traj[11][0], traj[11][1]
            if self.powerups:
                px, py = min(self.powerups, key=lambda p: math.hypot(p[0] - x, p[1] - y))[:2]
                cost += 0.002 * math.hypot(px - x, py - y)
            else:
                cost += HOME_W * (abs(x - self.home_x) + 0.25 * abs(y - H / 2))
            if d != self.cmd:
                cost += 0.01
            if best is None or cost < best:
                best, choice = cost, d
        return choice

    @staticmethod
    def _cost(traj, threats):
        cost = 0.0
        for i in range(2, len(traj), 3):
            t = (i + 1) * DT
            x, y = traj[i][0], traj[i][1]
            w = 1.0 / (1.0 + 2.0 * t)
            edge = min(x - LO, HI_X - x, y - LO, HI_Y - y)
            if edge < 80:
                cost += 0.02 * w * (80 - edge) / 80
            far = i % 12 == 11
            for tx, ty, vx, vy, r, m0, mt, ship in threats:
                d = math.hypot(tx + vx * t - x, ty + vy * t - y) - r
                if far and ship:
                    cost += STANDOFF * w * math.exp(-max(d, 0.0) / STANDOFF_PX)
                d -= m0 + mt * t
                if d < 40:
                    cost += w * math.exp(-max(d, 0.0) / 12.0) * (3.0 if d < 0 else 1.0)
        return cost


In [ ]:
%%writefile /kaggle/working/solvers/blarglike.py
"""Blarglike: parse the 30x20 grid from pixels, plan a battery-aware tour of flowers and
batteries, then take the exit.

The game canvas (1024x704, 32 px tiles, map offset 32,32) is scaled to fit the
viewport height and centred. A move animates in 2 frames and presses every 2
frames all register, so each decision sends up to MAX_MOVES moves as one timed
program and skips until it has played out; the frame after it shows the settled
grid. Every move (a wall bump too) costs 1 battery, so battery, not time, limits
the run. Flowers ("enemies" in the code) step randomly after every player move;
walking into one collects it. The battery is read from the HUD; at 0, R restarts.
"""
from __future__ import annotations

import io
from collections import deque

import numpy as np
from PIL import Image

from gga.protocol import Action

COLS, ROWS = 30, 20
MAX_MOVES = 8
BATTERY_MAX = 50
BATTERY_GAIN = 5
FLOWER_GAIN = 10
RESERVE = 20
CHASE = 2
PURSUIT = 6
GLYPHS = {
    " ": "0000000000000000000000000000000000000000",
    "/": "0000000000000040300806010040200804010080",
    "0": "0000000000000001c0c823084210842108c32070",
    "1": "0000000000000001c0f00c0300c0300c0300c0fc",
    "2": "0000000000000001c0880300c020180c060301fc",
    "3": "0000000000000001e088030080c01803004430f0",
    "4": "0000000000000000603816058260987f0180603c",
    "5": "0000000000000003e080200803e08c01004031f0",
    "6": "0000000000000000f060300803e0cc2108433078",
    "7": "0000000000000003f00402008060100403008020",
    "8": "0000000000000001e08c2108c1c08c2108423078",
    "9": "0000000000000001c0882308c2307403008060e0",
    "|": "0000000000080200802008020080200802008020",
}
TEMPLATES = {ch: np.unpackbits(np.frombuffer(bytes.fromhex(h), np.uint8))[:160].reshape(16, 10).astype(bool)
             for ch, h in GLYPHS.items()}
KEYS = {(0, -1): "ArrowUp", (0, 1): "ArrowDown", (-1, 0): "ArrowLeft", (1, 0): "ArrowRight"}


class Grid:
    """Per-tile sample indices for the map rectangle found on screen."""

    def __init__(self, left, top, tile_w, tile_h, width, height):
        self.box = (left, top, tile_w, tile_h)
        k = np.linspace(0.12, 0.88, 12)
        xs = np.array([[left + (c + f) * tile_w for f in k] for c in range(COLS)])
        ys = np.array([[top + (r + f) * tile_h for f in k] for r in range(ROWS)])
        self.xi = np.clip(xs.round().astype(int), 0, width - 1)
        self.yi = np.clip(ys.round().astype(int), 0, height - 1)

    @classmethod
    def find(cls, rgb):
        """The map's 30x20 tiles from the frame: the rectangle of navy/blue pixels (map
        floors and walls; the canvas around it is black, the HUD text white), or None."""
        h, w, _ = rgb.shape
        r, b = rgb[..., 0].astype(np.int16), rgb[..., 2].astype(np.int16)
        mask = (b > 40) & (r < 160)
        cols = np.nonzero(mask.mean(0) > 0.5)[0]
        rows = np.nonzero(mask.mean(1) > 0.5)[0]
        if len(cols) < COLS * 4 or len(rows) < ROWS * 4:
            return None
        x0, x1, y0, y1 = cols[0], cols[-1] + 1, rows[0], rows[-1] + 1
        tw, th = (x1 - x0) / COLS, (y1 - y0) / ROWS
        if not 0.85 < tw / th < 1.15:
            return None
        return cls(x0, y0, tw, th, w, h)

    def parse(self, rgb):
        t = rgb[self.yi[:, None, :, None], self.xi[None, :, None, :]]
        t = t.reshape(ROWS, COLS, -1, 3).astype(np.int16)
        r, g, b = t[..., 0], t[..., 1], t[..., 2]
        n = t.shape[2]
        yel = ((r > 200) & (g > 180) & (b < 140)).sum(-1) / n
        red = (r > 180) & (g > 80) & (g < 170)
        nred = red.sum(-1) / n
        redb = np.where(red, b, 0).sum(-1) / np.maximum(red.sum(-1), 1)
        lite = ((b > 200) & (r > 130)).sum(-1) / n
        blue = (b > 130).sum(-1) / n
        grid = np.full((ROWS, COLS), ".")
        grid[blue > 0.3] = "#"
        grid[lite > 0.25] = "X"
        grid[(nred > 0.03) & (redb <= 100)] = "A"
        grid[(nred > 0.03) & (redb > 100)] = "E"
        player = None
        if yel.max() > 0.04:
            player = tuple(int(v) for v in np.unravel_index(yel.argmax(), yel.shape)[::-1])
            grid[player[1], player[0]] = "@"
        return grid, player


def read_hud(gray, box):
    """The HUD's "Battery: 40 / 50" as (40, 50), or None when it cannot be read.

    The text is drawn at canvas (10, 10) in 7 px cells, one map tile left of and above the
    map (the map sits at canvas (32, 32)); the glyph templates fit the 1600x1000 viewport's
    scale only, so other scales read nothing (the solver then estimates its battery)."""
    left, top, tw, th = box
    s = tw / 32
    if abs(s - 1000 / 704) > 0.02:
        return None
    cx, cy = left - 32 * s, top - 32 * s
    best = None
    for dy in (-1, 0, 1):
        for dx in (-2, -1, 0, 1, 2):
            y0 = int(round(cy + 10 * s)) - 8 + dy
            text, total = "", 0
            for i in range(9, 17):
                x0 = int(round(cx + (10 + 7 * i) * s)) - 3 + dx
                cell = gray[max(y0, 0):y0 + 16, max(x0, 0):x0 + 10] > 110
                if cell.shape != (16, 10):
                    return None
                ch, err = min(((c, int((t != cell).sum())) for c, t in TEMPLATES.items()), key=lambda e: e[1])
                text, total = text + ch, total + err
            if best is None or total < best[0]:
                best = (total, text)
    total, text = best
    if total > 8 * 10:
        return None
    parts = text.split("/")
    if len(parts) < 2 or not parts[0].strip().isdigit():
        return None
    top_part = parts[1].strip().split(" ")[0]
    return int(parts[0].strip()), (int(top_part) if top_part.isdigit() and len(top_part) >= 2 else None)


def bfs(grid, start):
    """Distances and parents over non-wall tiles; the exit is a goal, never passed through."""
    dist = {start: 0}
    parent = {}
    q = deque([start])
    while q:
        x, y = q.popleft()
        if grid[y, x] == "X" and (x, y) != start:
            continue
        for dx, dy in KEYS:
            nx, ny = x + dx, y + dy
            if 0 <= nx < COLS and 0 <= ny < ROWS and grid[ny, nx] != "#" and (nx, ny) not in dist:
                dist[(nx, ny)] = dist[(x, y)] + 1
                parent[(nx, ny)] = (x, y)
                q.append((nx, ny))
    return dist, parent


def path_to(parent, start, goal):
    path = [goal]
    while path[-1] != start:
        path.append(parent[path[-1]])
    return path[::-1][1:]


def chase(grid, me, fl, depth=3):
    """The move that catches a wandering flower soonest in expectation.

    Expectimax over my move and the flower's reply: after each player move (a wall bump
    leaves the player in place) the flower steps uniformly to a free neighbouring floor
    tile, if it has one; moving into it catches it. Leaves are scored by the distance
    left, plus one when it is even (an even distance cannot close without a bump)."""
    dists = {}

    def d(a, b):
        if b not in dists:
            dists[b] = bfs(grid, b)[0]
        return dists[b].get(a, 99)

    def free(p, me):
        x, y = p
        return 0 <= x < COLS and 0 <= y < ROWS and grid[y, x] in ".@" and p != me

    def value(me, fl, depth):
        if depth == 0:
            h = d(me, fl)
            return h + (h % 2 == 0)
        best = None
        for dx, dy in KEYS:
            nxt = (me[0] + dx, me[1] + dy)
            if nxt == fl:
                return 1
            if not (0 <= nxt[0] < COLS and 0 <= nxt[1] < ROWS) or grid[nxt[1], nxt[0]] == "#":
                nxt = me
            elif grid[nxt[1], nxt[0]] == "X":
                continue
            moves = [(fl[0] + ex, fl[1] + ey) for ex, ey in KEYS if free((fl[0] + ex, fl[1] + ey), nxt)]
            v = 1 + sum(value(nxt, f, depth - 1) for f in moves or [fl]) / max(len(moves), 1)
            best = v if best is None else min(best, v)
        return best if best is not None else 99

    def score(k):
        dx, dy = k
        nxt = (me[0] + dx, me[1] + dy)
        if nxt == fl:
            return -1
        if not (0 <= nxt[0] < COLS and 0 <= nxt[1] < ROWS) or grid[nxt[1], nxt[0]] == "#":
            nxt = me
        elif grid[nxt[1], nxt[0]] == "X":
            return 999
        moves = [(fl[0] + ex, fl[1] + ey) for ex, ey in KEYS if free((fl[0] + ex, fl[1] + ey), nxt)]
        return sum(value(nxt, f, depth - 1) for f in moves or [fl]) / max(len(moves), 1)

    return min(KEYS, key=score)


class Solver:
    def __init__(self):
        self.grid = None
        self.candidate = None
        self.turn = 0
        self.idle = 0
        self.battery = None
        self.bmax = BATTERY_MAX
        self.gap = 1
        self.sent = None
        self.stalls = 0

    def step(self, obs, tools):
        self.turn += 1
        if self.turn == 1:
            return Action("press_key", {"key": "Space"})
        if obs.program_left_s > 0.05:
            return Action("skip", {})
        img = Image.open(io.BytesIO(obs.image))
        rgb = np.asarray(img.convert("RGB"))
        if self.grid is None:
            found = Grid.find(rgb)
            if found and self.candidate and all(abs(a - b) <= 1 for a, b in zip(found.box, self.candidate.box)):
                self.grid = found
            self.candidate = found
            if self.grid is None:
                return self._idle(None)
        hud = read_hud(np.asarray(img.convert("L")), self.grid.box)
        if hud:
            self.battery = hud[0]
            self.bmax = hud[1] or self.bmax
        grid, player = self.grid.parse(rgb)
        self._check(player)
        if player is None or self.battery == 0 or (hud is None and self.stalls >= 3):
            return self._idle(hud)
        b = self.battery if self.battery is not None else self.bmax
        moves = self._plan(grid, player, b)
        if not moves:
            return self._idle(hud)
        self.idle = 0
        end, moved = player, False
        for dx, dy in moves:
            nxt = (end[0] + dx, end[1] + dy)
            if not (0 <= nxt[0] < COLS and 0 <= nxt[1] < ROWS) or grid[nxt[1], nxt[0]] in "#E":
                break
            end, moved = nxt, True
        self.sent = (player, end, moved)
        if hud is None and self.battery is not None:
            self.battery = max(self.battery - len(moves), 0)
        steps = []
        for dx, dy in moves:
            steps.append({"keys": [KEYS[(dx, dy)]], "duration_frames": 1})
            steps.append({"keys": [], "duration_frames": self.gap})
        steps.append({"keys": [], "duration_frames": 0})
        return Action("sequence", {"steps": steps}, {"battery": b, "player": list(player), "gap": self.gap})

    def _check(self, player):
        """Compare where the last program should have left the player with where it is: still
        at the start means its presses were dropped (press slower) or the run is over."""
        if self.sent is None:
            return
        start, end, moved = self.sent
        self.sent = None
        if player == start and moved:
            self.stalls += 1
            if self.stalls >= 2:
                self.gap = min(self.gap + 1, 4)
        elif player is not None:
            self.stalls = 0

    def _idle(self, hud):
        """The title screen ("any key"), a finished run (R restarts it) or a transition."""
        self.idle += 1
        if self.idle % 5 == 1 and self.turn > 2:
            over = self.battery == 0 or self.stalls >= 3
            if over and self.idle > 1:
                self.stalls = 0
                self.battery = None
            return Action("press_key", {"key": "r" if over else "Space"})
        return Action("skip", {})

    def _plan(self, grid, player, b):
        dist, parent = bfs(grid, player)
        exits = [p for p in dist if grid[p[1], p[0]] == "X"]
        items = [p for p in dist if grid[p[1], p[0]] in "EA"]
        if not exits and not items:
            return []
        goal = self._tour(grid, player, b, dist, items, min(exits, key=dist.get) if exits else None)
        flower = grid[goal[1], goal[0]] == "E"
        if flower and dist[goal] <= PURSUIT:
            return [chase(grid, player, goal)]
        path = path_to(parent, player, goal)
        n = max(1, dist[goal] - 3) if flower else MAX_MOVES
        moves = []
        prev = player
        for p in path[:n]:
            moves.append((p[0] - prev[0], p[1] - prev[1]))
            prev = p
        return moves

    @staticmethod
    def _walls_near(grid, x, y):
        return sum(not (0 <= x + dx < COLS and 0 <= y + dy < ROWS) or grid[y + dy, x + dx] == "#"
                   for dx, dy in KEYS)

    def _tour(self, grid, player, b, dist, items, ex):
        """The first item of the best order to visit a subset of items and then the exit.

        Plans are ranked by (battery left at the exit capped at RESERVE, flowers taken, battery
        left): keep enough charge for the next floor, then take every flower that fits."""
        n = len(items)
        flower = [grid[p[1], p[0]] == "E" for p in items]

        extra = [(1 + CHASE) if flower[i] and dist[p] % 2 == 0 else (CHASE if flower[i] else 0)
                 for i, p in enumerate(items)]
        from_item = [bfs(grid, p)[0] for p in items]
        big = 10 ** 6

        def d(a, j):
            src = dist if a < 0 else from_item[a]
            return src.get(items[j], big) + extra[j]

        def collect(h, k, j):
            h -= k - 1
            if h <= 0:
                return None
            g = FLOWER_GAIN if flower[j] else BATTERY_GAIN
            return (self.bmax if h + g >= self.bmax else h + g + 1) - 1

        best = {}
        frontier = {(0, -1): b}
        first = {(0, -1): None}
        plans = []
        while frontier:
            nxt = {}
            for (mask, last), h in frontier.items():
                end = (dist if last < 0 else from_item[last]).get(ex, big) if ex else big
                nf = sum(flower[i] for i in range(n) if mask >> i & 1)
                left = h - end
                if left >= 1:
                    plans.append(((min(left, RESERVE), nf, left), first[(mask, last)]))
                else:
                    plans.append(((left - big, nf, 0), first[(mask, last)]))
                for j in range(n):
                    if mask >> j & 1:
                        continue
                    h2 = collect(h, d(last, j), j)
                    if h2 is None:
                        continue
                    key = (mask | 1 << j, j)
                    if h2 > nxt.get(key, -1) and h2 > best.get(key, -1):
                        nxt[key] = best[key] = h2
                        first[key] = first[(mask, last)] if last >= 0 else j
            frontier = nxt
        _, j = max(plans, key=lambda e: e[0])
        if j is None:
            return ex if ex else min(items, key=dist.get)
        return items[j]


In [ ]:
%%writefile /kaggle/working/solvers/cavebots.py
"""Cavebots: seal the Core into a dug pocket behind a barricade so mutants can never reach it.

The game (quasilyte/cavebots-game, Ebitengine) scores how long the Core survives. Mutants path with
A* over the 32 px cave grid; buildings block their cell. So the Core digs a short pocket off a flat
("hard") cell next to the starting tunnel, walks to its far end, and a barricade built on the flat
cell cuts the pocket off. Attackers then stop at the reachable cell closest to the Core, which the
pocket keeps two cells away with rock in between (no melee reach, no line of fire).

Screen geometry (1600x900 viewport, 960x540 world camera, so 5/3 px per world px): the camera
centres on the spawn cell, so world x 768 is screen x 0 and the tunnel row is found from pixels.
The 4 iron that pays for the barricade lies next to the Core; the harvester bot that carries it is
always the first cell the game revealed below the tunnel, found by replaying the reveal order.
"""
from __future__ import annotations

import io
import os

import numpy as np
from PIL import Image

from gga.protocol import Action

SCALE = 5 / 3
CELL = 32 * SCALE
CAM_X = 768
NEUTRAL = (1200, 150)
TUNNEL = list(range(39, 31, -1))


KINDS = {
    "hidden": (47, 52, 74), "base": (96, 104, 149), "weak": (83, 89, 123),
    "iron": (151, 146, 167), "unit": (87, 126, 94), "flat": (67, 55, 122),
    "rocky": (120, 83, 90), "mud": (40, 40, 71), "hard": (63, 57, 98), "hard": (63, 57, 98), "outer": (130, 125, 119), "grass": (2, 80, 2),
}
MOUNTAIN = {"hidden", "base", "weak", "iron", "unit", "flat", "rocky"}
SAFE_UNTIL = 48.0

LOG = os.environ.get("CAVEBOTS_LOG")
DUMP = os.environ.get("CAVEBOTS_DUMP")
DUMP = os.environ.get("CAVEBOTS_DUMP")


def cell_xy(cx, cy, ytop, r):
    """Screen centre of grid cell (cx, cy) given the tunnel row r's top screen y."""
    return int((cx * 32 + 16 - CAM_X) * SCALE), int(ytop + (cy - r + 0.5) * CELL)


class Solver:
    def __init__(self):
        self.t0 = None
        self.battle_since = None
        self.plan = None
        self.step_i = 0
        self.wait_until = 0.0
        self.last_try = -99.0
        self.seal_cell = None
        self.last_seal = -99.0
        self.confirm = 0
        self.stood = False
        self.presses = 0
        self.branch, self.dug_branch, self.burst_since, self.down, self.fled = None, 0, None, 0, False
        self.battles = 0
        self.scene = None


    def log(self, t, msg):
        if LOG:
            with open(LOG, "a") as f:
                f.write(f"{t:7.2f} {msg}\n")

    @staticmethod
    def detect_scene(im):
        grass = im[60:780:60, 1000:1580:60].reshape(-1, 3)
        if (np.mean((grass[:, 1] > 60) & (grass[:, 0] < 40)) > 0.5):
            return "battle"
        def cyan(y0, y1):
            band = im[y0:y1, 600:1000].reshape(-1, 3)
            return np.mean((band[:, 2] > 150) & (band[:, 0] < 150))
        if cyan(250, 300) > 0.01:
            return "results"
        if cyan(110, 170) > 0.01:
            return "menu"
        if cyan(70, 125) > 0.01:
            return "settings"
        return "other"

    def cell_mean(self, im, cx, cy):
        x, y = cell_xy(cx, cy, self.ytop, self.r)
        return im[y - 15:y + 15, x - 15:x + 15].reshape(-1, 3).mean(0)

    def classify(self, im, cx, cy):
        c = self.cell_mean(im, cx, cy)
        return min(KINDS, key=lambda k: float(np.sum((c - np.array(KINDS[k])) ** 2)))

    def find_tunnel(self, im):
        """Top screen y of the tunnel row: the gap in the beige outer column (cell x 39)."""
        x = int((39 * 32 + 16 - CAM_X) * SCALE)
        col = im[:800, x - 10:x + 10].mean(1)
        beige = (np.abs(col - np.array(KINDS["outer"])).sum(1) < 60)
        gap = np.where(~beige)[0]
        if len(gap) < 30:
            return None

        runs = np.split(gap, np.where(np.diff(gap) > 1)[0] + 1)
        run = max(runs, key=len)
        centre = (run[0] + run[-1]) / 2
        return centre - CELL / 2


    def infer_harvester(self, kinds, r):
        """Replay the game's reveal order (randIterate over the tunnel) to find the seq-0 cell.
        Fixed loot: seq 0 harvester, 3 flat, 6 iron, 8 patrol, 11 generator bot."""
        need = {0: "unit", 3: "flat", 6: "iron", 8: "unit", 11: "unit"}
        found = set()
        for start in range(8):
            for inc in (True, False):
                seen, order, i = set(), [], start
                for _ in range(8):
                    x = TUNNEL[i]
                    for c in ((x + 1, r), (x, r + 1), (x - 1, r), (x, r - 1)):
                        if c in kinds and c not in seen:
                            seen.add(c)
                            order.append(c)
                    i = (i + 1) % 8 if inc else (i - 1) % 8
                if all(s < len(order) and kinds[order[s]] == k for s, k in need.items()):
                    found.add((order[0], order[6]))
        return found

    def make_plan(self, im):
        r = self.r
        kinds = {}
        for x in range(32, 39):
            for dy in (-1, 1):
                kinds[(x, r + dy)] = self.classify(im, x, r + dy)
        kinds[(31, r)] = self.classify(im, 31, r)
        found = self.infer_harvester(kinds, r)
        if len(found) == 1:
            (h, _), = found
            harv = {h}
            options = [{h}, set()]
        else:
            harv = {c for c, k in kinds.items() if k == "unit" and c[1] != r}
            options = [harv, set()]
        flats = [c for c, k in kinds.items() if k == "flat"]
        best = None

        for hs in options:
            for f in flats:
                plan = self.plan_pocket(f, hs, kinds)
                if plan and (best is None or plan[0] < best[0]):
                    best = plan
            if best:
                break
        self.log(0, f"r={r} kinds={kinds} harv={harv} flats={flats} best={best}")
        if best is None:
            return None
        cost, steps, f = best
        return steps

    def plan_pocket(self, f, harv, kinds):
        """BFS a pocket from flat cell f through rock to a resting cell P for the Core.

        Attack orders aim at the Core's position, 40% of the time offset by up to 64 px (2 cells).
        The A* then stops at the reachable cell nearest that target, so for every target within
        2 cells of P the nearest reachable cells must sit 2+ cells from both P and the plug f."""
        r = self.r
        if any(max(abs(h[0] - f[0]), abs(h[1] - f[1])) <= 1 for h in harv):
            return None
        open_ = {(x, r) for x in range(32, 40)} | set(harv)

        def reachable(c):
            return c in open_ or c[0] >= 40

        def ok_cell(c):
            x, y = c
            if not (1 <= x <= 38 and r - 8 <= y <= r + 8) or c == f or reachable(c):
                return False
            return not any(reachable((x + dx, y + dy)) for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)))

        def cheb(a, b):
            return max(abs(a[0] - b[0]), abs(a[1] - b[1]))

        cands = [(x, y) for x in range(28, 44) for y in range(r - 11, r + 12) if reachable((x, y))]

        def nearest(p):
            d = min(abs(x - p[0]) + abs(y - p[1]) for x, y in cands)
            return [c for c in cands if abs(c[0] - p[0]) + abs(c[1] - p[1]) == d]


        clear = {c for c in cands if c[0] >= 40}
        todo = list(clear)
        while todo:
            x, y = todo.pop()
            for n in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if n in open_ and n not in clear and cheb(n, f) >= 2:
                    clear.add(n)
                    todo.append(n)

        def safe(p):
            if any(cheb(n, p) < 2 for n in nearest(p)):
                return False
            for dx in range(-2, 3):
                for dy in range(-2, 3):
                    if any(n not in clear for n in nearest((p[0] + dx, p[1] + dy))):
                        return False
            return True

        start = [c for c in ((f[0] + 1, f[1]), (f[0] - 1, f[1]), (f[0], f[1] + 1), (f[0], f[1] - 1))
                 if ok_cell(c)]
        prev = {c: None for c in start}
        frontier = list(start)
        best = None
        while frontier and best is None:
            nxt = []
            for c in frontier:
                if safe(c):
                    path = []
                    p = c
                    while p is not None:
                        path.append(p)
                        p = prev[p]
                    path.reverse()
                    if len(path) >= 2 and (best is None or len(path) < len(best)):
                        best = path
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    n = (c[0] + dx, c[1] + dy)
                    if n not in prev and ok_cell(n):
                        prev[n] = c
                        nxt.append(n)
            frontier = nxt
        if best is None:
            return None
        path = best
        steps = [("dig", h) for h in sorted(harv, key=lambda h: (kinds.get(h) != "unit", abs(h[0] - 36)))]
        steps += [("dig", f), ("dig", path[0]), ("dig", path[1]), ("seal", f)]
        steps += [("dig", c) for c in path[2:]] + [("move", path[-1])]
        return len(harv) + 1 + len(path), steps, f


    def click(self, x, y, button="left"):
        nx, ny = NEUTRAL
        return Action("sequence", {"steps": [
            {"x": x, "y": y, "duration_frames": 1},
            {"x": x, "y": y, "buttons": [button], "duration_frames": 2},
            {"x": x, "y": y, "duration_frames": 1},
            {"x": nx, "y": ny, "duration_frames": 0},
        ]})

    def press_at(self, x, y, key):
        nx, ny = NEUTRAL
        return Action("sequence", {"steps": [
            {"x": x, "y": y, "duration_frames": 2},
            {"x": x, "y": y, "keys": [key], "duration_frames": 2},
            {"x": x, "y": y, "duration_frames": 1},
            {"x": nx, "y": ny, "duration_frames": 0},
        ]})

    def walled(self, im, c):
        """A barricade's bright outline covers ~40% of its cell; a bare flat cell has none."""
        x, y = cell_xy(c[0], c[1], self.ytop, self.r)
        a = im[y - 18:y + 18, x - 18:x + 18].reshape(-1, 3).astype(int)
        return float(np.mean(a.sum(1) > 350)) > 0.25

    def branch_cells(self):
        """A 1-wide serpentine off the Core's resting cell R = (38, row): along R's row, then rows 2 and
        4 cells toward the tunnel, between columns 26 and 36, cut at the first cell that would touch
        open ground other than its predecessor or come within 2 cells of anything mutants reach."""
        kind, rest = self.plan[-1]
        if kind != "move" or rest[0] != 38:
            return []
        r, y0 = self.r, rest[1]
        s = 1 if y0 > r else -1
        rows = [y0, y0 - 2 * s, y0 - 4 * s]
        cells = [(37, y0)]
        for i, y in enumerate(rows):
            xs = range(36, 25, -1) if i % 2 == 0 else range(26, 37)
            cells += [(x, y) for x in xs]
            if i + 1 < len(rows):
                cells.append((cells[-1][0], y - s))
        digs = {c for k, c in self.plan if k == "dig"}
        open_ = {(x, r) for x in range(32, 40)} | digs
        reach = {(x, r) for x in range(32, 40)} | {c for c in digs if abs(c[1] - r) <= 1 and c != self.seal_cell}
        out, prev = [], rest
        for c in cells:
            nbrs = [(c[0] + dx, c[1] + dy) for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1))]
            x, y = cell_xy(c[0], c[1], self.ytop, r)
            if (c in open_ or not (40 <= y <= 770 and x >= 40)
                    or any(n in open_ and n != prev for n in nbrs)
                    or min(max(abs(c[0] - q[0]), abs(c[1] - q[1])) for q in reach) < 2
                    or max(abs(c[0] - self.seal_cell[0]), abs(c[1] - self.seal_cell[1])) < 2):
                break
            out.append(c)
            open_.add(c)
            prev = c
        return out

    def extend(self, im, t):
        """After sealing, until SAFE_UNTIL: dig a retreat branch while energy lasts, then return to R
        (attacks aim where the Core is, and R's nearest reachable cell is the grass). When the plug
        falls, run to the branch's far end: mutants must then walk the whole branch."""
        if self.branch is None:
            self.branch = self.branch_cells()
            self.dug_branch = 0
            self.log(t, f"branch {len(self.branch)} cells: {self.branch[:3]}...")
        if not self.branch:
            return Action("skip", {})
        rest = self.plan[-1][1]
        walled = self.walled(im, self.seal_cell)
        self.down = 0 if walled else self.down + 1
        if self.stood and self.down >= 10 and self.dug_branch and not self.fled:
            self.fled = True
            end = self.branch[self.dug_branch - 1]
            self.log(t, f"plug down: fleeing to {end}")
            self.burst_since, self.last_try, self.confirm = t, t, 0
            x, y = cell_xy(end[0], end[1], self.ytop, self.r)
            return self.click(x, y, "right")

        early = t - self.battle_since < SAFE_UNTIL or self.fled
        if self.burst_since is None:
            if early and self.dug_branch < len(self.branch):
                self.burst_since, self.last_try, self.confirm = t, -99.0, 0
            return Action("skip", {})
        if self.dug_branch >= len(self.branch):
            return Action("skip", {})
        c = self.branch[self.dug_branch]
        k = self.classify(im, *c)
        self.confirm = self.confirm + 1 if k not in MOUNTAIN and t - self.last_seal > 2.0 else 0
        if self.confirm >= 3:
            self.confirm = 0
            self.dug_branch += 1
            self.log(t, f"branch dug {c} -> {k} ({self.dug_branch}/{len(self.branch)})")
            self.last_try = -99.0
            if self.dug_branch < len(self.branch):
                return Action("skip", {})
        elif early and (self.last_try < 0 or t - self.last_try < 4.0 or self.fled):
            if self.last_try < 0 or (self.fled and t - self.last_try > 3.0):
                self.last_try = t
                x, y = cell_xy(c[0], c[1], self.ytop, self.r)
                return self.click(x, y)
            return Action("skip", {})
        if self.fled:
            return Action("skip", {})

        self.burst_since = None
        x, y = cell_xy(rest[0], rest[1], self.ytop, self.r)
        return self.click(x, y, "right")

    def step(self, obs, tools):
        t = obs.elapsed_s
        im = np.asarray(Image.open(io.BytesIO(obs.image)).convert("RGB"))
        scene = self.detect_scene(im)
        if DUMP and int(t * 5) % 10 == 0:
            Image.fromarray(im).save(f"{DUMP}/t{t:06.1f}.jpg")
        if scene != self.scene:
            self.log(t, f"scene {scene}")
            self.scene = scene
        if scene != "battle":
            self.battle_since = None
            if t < self.wait_until:
                return Action("skip", {})
            self.wait_until = t + 1.0
            if scene == "results":
                return self.click(800, 595)
            if scene == "menu":
                return self.click(800, 335)
            if scene == "settings":
                return self.click(800, 770)
            return Action("skip", {})

        if self.battle_since is None:
            self.battle_since = t
            self.plan, self.step_i, self.seal_cell, self.last_seal = None, 0, None, -99.0
            self.stood, self.presses = False, 0
            self.branch, self.dug_branch, self.burst_since, self.down, self.fled = None, 0, None, 0, False
            self.battles += 1
            self.log(t, f"battle {self.battles} starts")
        if t - self.battle_since < 0.6:
            return Action("skip", {})
        if self.plan is None:
            ytop = self.find_tunnel(im)
            if ytop is None:
                return Action("skip", {})
            self.ytop = ytop
            self.r = 0
            self.plan = self.make_plan(im) or []
            self.log(t, f"plan {self.plan}")


        if self.seal_cell and self.walled(im, self.seal_cell):
            self.stood = True
        elif self.seal_cell and t - self.last_seal > 2.5 and not self.fled:
            self.last_seal = t
            sx, sy = cell_xy(self.seal_cell[0], self.seal_cell[1], self.ytop, self.r)


            self.presses += 1
            key = "q" if self.stood and self.presses % 2 == 0 else "w"
            return self.press_at(sx, sy, key)
        if self.step_i >= len(self.plan):
            return self.extend(im, t)

        kind, c = self.plan[self.step_i]
        if kind == "seal":
            self.seal_cell = c
            self.log(t, f"sealing {c}")
            self.step_i += 1
            return self.step(obs, tools)
        x, y = cell_xy(c[0], c[1], self.ytop, self.r)
        if kind == "dig":
            k = self.classify(im, *c)


            self.confirm = self.confirm + 1 if k not in MOUNTAIN and t - self.last_seal > 2.0 else 0
            if self.confirm >= 3:
                self.confirm = 0
                self.log(t, f"dug {c} -> {k}")
                self.step_i += 1
                self.last_try = -99
                return self.step(obs, tools) if self.step_i < len(self.plan) else Action("skip", {})
            if t - self.last_try > 2.0:
                self.last_try = t
                return self.click(x, y)
            return Action("skip", {})
        if kind == "move":
            if self.last_try < 0:
                self.last_try = t
                return self.click(x, y, "right")
            if t - self.last_try > 2.0:
                self.step_i += 1
                self.last_try = -99
            return Action("skip", {})
        return Action("skip", {})


In [ ]:
%%writefile /kaggle/working/solvers/generic.py
"""Fallback for unseen games: the starter template's opener, then hold right."""
from gga.protocol import Action

OPENING = ("Space", "Enter", "click")


class Solver:
    def __init__(self):
        self.turn = 0

    def step(self, obs, tools):
        self.turn += 1
        if self.turn <= len(OPENING):
            opener = OPENING[self.turn - 1]
            if opener == "click":
                return Action("click", {"x": obs.info["width"] // 2, "y": obs.info["height"] // 2})
            return Action("press_key", {"key": opener})
        return Action("sequence", {"steps": [{"keys": ["ArrowRight"], "duration_ms": 0}]})


In [ ]:
%%writefile /kaggle/working/solvers/off_line.py
"""Off Line: stay on the line, dodge orange squares, collect blue ones.

Rules from the game's source: a 512x512 canvas, a 32 px player moving 4 px a
game tick per held arrow, a 5 px line, squares falling speed px a tick
(truncated to whole px) with speed = 4 * 1.0001**ticks since the attempt
began. When and where is perceived, not assumed: the canvas and the line from
the pixels, the attempt's start from where the Space press landed
(last_action), game ticks per frame from the player's measured step, and the
delay until a reply lands from last_action. The speed rule predicts the
squares' fall and is checked every turn against their fall measured between
obs.frames: after consistent disagreement it is re-anchored (its tick count
inverted from the measured speed), or replaced by the measurement if nothing
fits. Each decision detects the squares and the player, then plans the next
frames by dynamic programming over a lattice of player positions one step
apart, scoring health lost (squares: 1 a tick, off the line: 0.2 a tick),
score lost off the line and health gained from blue squares.
"""
from __future__ import annotations

import io
from collections import deque

import numpy as np
from PIL import Image

from gga.protocol import Action

W = 512
PLAYER = 32
PLAYER_TICK = 4
LINE_HW = 2.5
SPEED0, GROWTH = 4.0, 1.0001
MISMATCHES = 3
NAVAJO = (255, 222, 173)
HORIZON = 24
REACH = 16

ACTIONS = [(0, 0), (-1, 0), (1, 0), (0, -1), (0, 1)]
KEYS = {(-1, 0): ["ArrowLeft"], (1, 0): ["ArrowRight"], (0, -1): ["ArrowUp"], (0, 1): ["ArrowDown"], (0, 0): []}


def components(mask: np.ndarray) -> list[tuple[int, int, int, int]]:
    """Bounding boxes (x0, y0, x1, y1), exclusive ends, of 4-connected blobs."""
    h, w = mask.shape
    seen = np.zeros_like(mask, dtype=bool)
    boxes = []
    for y, x in zip(*np.nonzero(mask)):
        if seen[y, x]:
            continue
        seen[y, x] = True
        q = deque([(y, x)])
        x0 = x1 = x
        y0 = y1 = y
        while q:
            cy, cx = q.popleft()
            x0, x1, y0, y1 = min(x0, cx), max(x1, cx), min(y0, cy), max(y1, cy)
            for ny, nx in ((cy + 1, cx), (cy - 1, cx), (cy, cx + 1), (cy, cx - 1)):
                if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] and not seen[ny, nx]:
                    seen[ny, nx] = True
                    q.append((ny, nx))
        boxes.append((x0, y0, x1 + 1, y1 + 1))
    return boxes


def decode(jpeg: bytes) -> np.ndarray:
    return np.asarray(Image.open(io.BytesIO(jpeg)).convert("RGB"), dtype=np.int16)


def orange_boxes(c: np.ndarray) -> list[tuple[int, int, int, int]]:
    """Orange squares in a W x W canvas crop, canvas px."""
    s = c[::2, ::2]
    r2, g2, b2 = s[..., 0], s[..., 1], s[..., 2]
    orange = (r2 > 200) & (g2 > 125) & (g2 < 205) & (b2 < 100)
    return [(2 * x0, 2 * y0, 2 * x1, 2 * y1) for x0, y0, x1, y1 in components(orange)
            if (x1 - x0) >= 6]


class Solver:
    def __init__(self):
        self.start_frame = None
        self.space_frame = -999
        self.plan = {}
        self.player = None
        self.ticks = 3.0
        self.steps_seen = []
        self.gap = 4
        self.sent_frame = None
        self.prev = None
        self.space_land = None
        self.tick0 = 0.0
        self.mismatch = 0
        self.anchors = 0
        self.measured = None
        self.rule_ok = True
        self.canvas = None
        self.line_cx = 256.0

    @property
    def step_px(self):
        return PLAYER_TICK * self.ticks

    def find_canvas(self, img):
        nav = ((np.abs(img[..., 0] - NAVAJO[0]) < 14) & (np.abs(img[..., 1] - NAVAJO[1]) < 14)
               & (np.abs(img[..., 2] - NAVAJO[2]) < 18))
        cols, rows = np.nonzero(nav.any(axis=0))[0], np.nonzero(nav.any(axis=1))[0]
        if len(cols) < 64 or len(rows) < 64:
            return self.canvas
        return int(cols[0]), int(rows[0]), int(max(cols[-1] + 1 - cols[0], rows[-1] + 1 - rows[0]))

    def crop(self, img):
        """The canvas, resampled to W x W canvas px (nearest pixel) if it is drawn scaled."""
        x0, y0, size = self.canvas
        c = img[y0:y0 + W, x0:x0 + W]
        if abs(size - W) <= 2 and c.shape[:2] == (W, W):
            return c
        idx = (np.arange(W) * size / W).astype(int)
        return img[np.clip(y0 + idx, 0, img.shape[0] - 1)][:, np.clip(x0 + idx, 0, img.shape[1] - 1)]


    def perceive(self, img):
        c = self.crop(img)
        green = (c[..., 1] > 170) & (c[..., 0] > 110) & (c[..., 0] < 200) & (c[..., 2] < 110)
        slate = (c[..., 2] > 170) & (c[..., 0] < 140) & (c[..., 1] < 120)
        line_cols = (green | slate).mean(axis=0)
        hits = np.nonzero(line_cols > 0.5)[0]
        playing = len(hits) > 0
        if playing:
            self.line_cx = float(hits.mean() + 0.5)
        s = c[::2, ::2]
        r2, g2, b2 = s[..., 0], s[..., 1], s[..., 2]
        blue = (b2 > 190) & (r2 < 110) & (g2 > 100) & (g2 < 190)
        slatep = (b2 > 170) & (r2 > 70) & (r2 < 140) & (g2 > 60) & (g2 < 120)
        black = (r2 < 70) & (g2 < 70) & (b2 < 70)
        obstacles = orange_boxes(c)
        body = blue | slatep | black
        lc = int(self.line_cx) // 2
        body[:, lc - 2:lc + 2] = False
        body[:32, :100] = False
        boxes = components(body)
        player = None
        big = [bx for bx in boxes if (bx[3] - bx[1]) >= 11]

        if big:
            merged = []
            for bx in sorted(big):
                if merged and bx[0] - merged[-1][2] <= 6 and bx[1] < merged[-1][3] and bx[3] > merged[-1][1]:
                    m = merged[-1]
                    merged[-1] = (min(m[0], bx[0]), min(m[1], bx[1]), max(m[2], bx[2]), max(m[3], bx[3]))
                else:
                    merged.append(bx)
            cands = [m for m in merged if 12 <= m[2] - m[0] <= 22 and 12 <= m[3] - m[1] <= 22]
            if cands:
                if self.player is not None:
                    px, py = self.player
                    m = min(cands, key=lambda m: abs(m[0] + m[2] - 16 - px) + abs(m[1] + m[3] - 16 - py))
                else:
                    m = max(cands, key=lambda m: m[3])
                cx, cy = (m[0] + m[2]), (m[1] + m[3])
                player = (cx - 16.0, cy - 16.0)
                inner_rows = (blue | slatep)[m[1]:m[3], m[0]:m[2]].any(axis=1).sum() * 2
                self.health = float(min(32, inner_rows))
        potions = []
        for x0, y0, x1, y1 in components(blue):
            w_, h_ = x1 - x0, y1 - y0
            if 6 <= w_ <= 10 and h_ <= 10 and not (y0 < 30 and x0 < 100):
                bx = (2 * x0, 2 * y0, 2 * x1, 2 * y1)
                if player is None or not (bx[0] >= player[0] - 4 and bx[2] <= player[0] + 36
                                          and bx[1] >= player[1] - 4 and bx[3] <= player[1] + 36):
                    potions.append(bx)
        return playing, player, obstacles, potions


    def measure_fall(self, obs, obstacles):
        """The squares' fall per frame: the same squares (same columns) in the oldest of obs.frames."""
        fps = obs.info.get("fps", 20)
        if len(obs.frames) < 2 or not obstacles:
            return None
        (t0, jpeg), t1 = obs.frames[0], obs.frames[-1][0]
        dt = (t1 - t0) * fps
        if dt < 0.5:
            return None
        before = orange_boxes(self.crop(decode(jpeg)))
        shifts = []
        for a in obstacles:
            best = None
            for b in before:
                if abs(a[0] - b[0]) > 2 or abs(a[2] - b[2]) > 2:
                    continue
                if a[3] < W - 2 and b[3] < W - 2:
                    d = a[3] - b[3]
                elif a[1] > 0 and b[1] > 0:
                    d = a[1] - b[1]
                else:
                    continue
                if d >= -1 and (best is None or d < best):
                    best = d
            if best is not None:
                shifts.append(best)
        if not shifts:
            return None
        return max(0.0, float(np.median(shifts))) / dt

    def learn(self, obs, f, player):
        """The landing delay of our last reply, and ticks per frame from the player's step."""
        la = obs.last_action or {}
        if self.sent_frame is not None and la.get("tool") == "sequence" and la.get("ok") and la.get("frame") is not None:
            self.gap = max(1, int(la["frame"]) - self.sent_frame)
        if player is not None and self.prev is not None:
            pf, ppx, ppy = self.prev
            held = [self.plan.get(k, (0, 0)) for k in range(pf, f)]
            nx, ny = sum(m[0] for m in held), sum(m[1] for m in held)
            hi = W - PLAYER

            xs, ys = [m[0] for m in held if m[0]], [m[1] for m in held if m[1]]
            if abs(nx) >= 2 and not ys and abs(nx) == len(xs) and 2 < player[0] < hi - 2 and 2 < ppx < hi - 2:
                self.steps_seen.append((player[0] - ppx) / nx)
            if abs(ny) >= 2 and not xs and abs(ny) == len(ys) and 2 < player[1] < hi - 2 and 2 < ppy < hi - 2:
                self.steps_seen.append((player[1] - ppy) / ny)
            self.steps_seen = [s_ for s_ in self.steps_seen if 1.0 <= s_ <= 80.0][-21:]
            if len(self.steps_seen) >= 5:
                t = float(np.median(self.steps_seen)) / PLAYER_TICK
                self.ticks = float(round(t)) if abs(t - round(t)) < 0.25 and round(t) >= 1 else t
        self.prev = (f, player[0], player[1]) if player is not None else None

    def speed_at(self, frame):
        """The rule's squares' speed (px a tick, before truncation) at an episode frame."""
        return SPEED0 * GROWTH ** (self.tick0 + max(0, frame - (self.start_frame or frame)) * self.ticks)

    def check_rule(self, f, fall):
        """Compare the measured fall with the rule's truncated speed; re-anchor the rule's
        tick count after consistent disagreement, or give it up if nothing fits."""
        per_tick = fall / self.ticks
        self.mismatch = self.mismatch + 1 if abs(per_tick - int(self.speed_at(f))) > 0.5 else 0
        if self.mismatch < MISMATCHES:
            return
        self.mismatch, self.anchors = 0, self.anchors + 1
        k = round(per_tick)
        if per_tick < SPEED0 - 0.5 or self.anchors > 4:
            self.rule_ok = False
            return

        target = k if abs(per_tick - k) < 0.3 else per_tick
        ticks = np.log(max(target, SPEED0) / SPEED0) / np.log(GROWTH)
        self.tick0 = ticks - max(0, f - self.start_frame) * self.ticks


    def step(self, obs, tools):
        f = obs.frame if obs.frame is not None else round(obs.elapsed_s * obs.info.get("fps", 20))
        img = decode(obs.image)
        self.canvas = self.find_canvas(img)
        if self.canvas is None:
            return Action("skip", {})
        playing, player, obstacles, potions = self.perceive(img)
        la = obs.last_action or {}
        if la.get("tool") == "press_key" and la.get("ok") and la.get("frame") is not None:
            self.space_land = int(la["frame"])
        if not playing:
            self.start_frame = None
            self.plan = {}
            self.player = self.prev = self.sent_frame = None
            if f - self.space_frame >= 2 * obs.info.get("grid_frames", 4):
                self.space_frame = f
                return Action("press_key", {"key": "Space"})
            return Action("skip", {})
        if self.start_frame is None:
            known = self.space_land is not None and self.space_land <= f
            self.start_frame = self.space_land if known else f
            self.tick0, self.mismatch, self.anchors, self.rule_ok = 0.0, 0, 0, True
        self.learn(obs, f, player)
        fall = self.measure_fall(obs, obstacles)
        if fall is not None:
            self.measured = fall if self.measured is None else 0.5 * self.measured + 0.5 * fall
            self.check_rule(f, fall)
        if player is None:
            player = self.player if self.player is not None else (W / 2 - PLAYER / 2, W - 2 * PLAYER)

        hi = W - PLAYER
        px, py = player
        for k in range(f, f + self.gap):
            dx, dy = self.plan.get(k, (0, 0))
            px = min(hi, max(0.0, px + dx * self.step_px))
            py = min(hi, max(0.0, py + dy * self.step_px))
        self.player = player
        health = getattr(self, "health", 32.0)
        moves = self.solve(f + self.gap, px, py, obstacles, potions, health)
        self.plan.update({f + self.gap + i: m for i, m in enumerate(moves)})
        self.sent_frame = f
        for k in [k for k in self.plan if k < f - 64]:
            del self.plan[k]
        steps, i = [], 0
        while i < len(moves):
            j = i
            while j < len(moves) and moves[j] == moves[i]:
                j += 1
            steps.append({"keys": KEYS[moves[i]], "duration_frames": j - i})
            i = j
        steps.append({"keys": [], "duration_frames": 0})
        return Action("sequence", {"steps": steps})

    def solve(self, f0, px, py, obstacles, potions, health):
        n = 2 * REACH + 1
        STEP, TICKS, GAP, hi = self.step_px, self.ticks, self.gap, W - PLAYER
        offs = (np.arange(n) - REACH) * STEP
        X = np.clip(px + offs[None, :], 0, hi) * np.ones((n, 1))
        Y = np.clip(py + offs[:, None], 0, hi) * np.ones((1, n))
        valid = ((px + offs[None, :] > -STEP) & (px + offs[None, :] < hi + STEP)
                 & (py + offs[:, None] > -STEP) & (py + offs[:, None] < hi + STEP))
        wh = 32.0 / max(health, 3.0)

        costs = []
        fall = 0.0
        rate = (self.speed_at if self.rule_ok or self.measured is None
                else lambda frame: self.measured / TICKS)
        speed0 = rate(f0 - GAP)
        for k in range(GAP):
            fall += speed0 * TICKS
        for t in range(HORIZON):
            sp = rate(f0 + t)
            fall += sp * TICKS
            cost = np.zeros((n, n))
            pcx, pcy = X + 16, Y + 16
            off = np.abs(pcx - self.line_cx) >= PLAYER / 2 + LINE_HW
            cost += off * (0.2 * TICKS * wh + 1.0)
            for x0, y0, x1, y1 in obstacles:
                hw, hh = (x1 - x0) / 2, max(y1 - y0, 16) / 2
                ocx = (x0 + x1) / 2
                ocy = (y1 - hh) + fall
                hit = (np.abs(pcx - ocx) < 16 + hw + 2) & (np.abs(pcy - ocy) < 16 + hh + 2)
                cost += hit * (1.0 * TICKS * wh * 1.5)
            for x0, y0, x1, y1 in potions:
                ocx, ocy = (x0 + x1) / 2, (y1 - 8) + fall
                got = (np.abs(pcx - ocx) < 24) & (np.abs(pcy - ocy) < 24)
                cost -= got * min(8.0, 32.0 - health) * wh * 0.5 / 3
            cost += np.maximum(0, hi * 0.75 - Y) * 0.01
            cost[~valid] = 1e6
            costs.append(cost)

        V = np.zeros((n, n))
        best = []
        for t in range(HORIZON - 1, -1, -1):
            cands = []
            for dx, dy in ACTIONS:
                sh = np.full((n, n), 1e7)
                ys = slice(max(0, -dy), n - max(0, dy))
                yd = slice(max(0, dy), n - max(0, -dy))
                xs = slice(max(0, -dx), n - max(0, dx))
                xd = slice(max(0, dx), n - max(0, -dx))
                sh[ys, xs] = (costs[t] + V)[yd, xd]
                cands.append(sh + 0.01 * (dx != 0 or dy != 0))
            cands = np.stack(cands)
            best.append(cands.argmin(axis=0))
            V = cands.min(axis=0)
        best.reverse()
        iy = ix = REACH
        moves = []
        for t in range(HORIZON):
            a = ACTIONS[best[t][iy, ix]]
            moves.append(a)
            ix = min(n - 1, max(0, ix + a[0]))
            iy = min(n - 1, max(0, iy + a[1]))
        return moves


In [ ]:
%%writefile /kaggle/working/scripted_agent.py
"""Submission agent for local runs: scripted solvers for the sample games, solvers.generic otherwise."""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

from solvers.dispatch import Agent


## The agent

The router: sample games to their solver, every other game to the start step + model agent.


In [ ]:
from solvers.dispatch import Agent as Dispatch

StartStepAgent = Agent


class Agent(Dispatch):
    """Sample games -> scripted solver; every other game -> start step + MODEL_AGENT."""
    def __init__(self):
        super().__init__(fallback=StartStepAgent())


## Submit

In [ ]:



from kaggle_evaluation.gga.gga_inference_server import GGAInferenceServer

if chromium.wait():
    print('warning: Chromium install returned an error')
MANAGER.wait_until(SERVE_AT_S)
print(f'serving at {time.time() - NOTEBOOK_START_TIME:.0f}s, model={MANAGER.state}')
Path('/kaggle/working/agent.py').write_text('from __main__ import Agent\n')
GGAInferenceServer().run()
